## 1\. Import libraries

In [1]:
# ---- Step 1: Import libraries ----
import pandas as pd
import os
import glob
import re

from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from datetime import datetime, date
from zoneinfo import ZoneInfo

# \(A\) QC FILE VALIDATION

## 2\. Load QC file \(raw, no changes yet\)

In [2]:
# ---- Step 2: Load QC file (raw) ----
qc_folder = "from_qcAnalyst"
qc_files = glob.glob(os.path.join(qc_folder, "qc_report_*.csv"))

if len(qc_files) == 0:
    print("No QC file found. Check the folder name or file name format.")
elif len(qc_files) > 1:
    print("More than one QC file found. Expected only one.")
    for f in qc_files:
        print(" -", f)
else:
    qc_path = qc_files[0]
    qc_df = pd.read_csv(qc_path)

    print(f"QC file loaded: {qc_path}")
    print(f"Rows: {len(qc_df)}")
    print(f"Columns: {list(qc_df.columns)}")

# ---- Step 2 (addendum): Drop blank spacer rows from raw QC export ----
# The QC export inserts a fully blank row between every real record.
# Uncomment the line below to drop them before validation runs.
# Comment it out again if you want to test validation against the raw, unfiltered file.

#qc_df = qc_df.dropna(subset=["id"]).reset_index(drop=True)
#print(f"Rows after dropping blank spacer rows: {len(qc_df)}")

QC file loaded: from_qcAnalyst\qc_report_20260701.csv
Rows: 12553
Columns: ['id', 'lot_number', 'product_code', 'customer', 'status', 'remarks', 'action', 'original_lot', 'evaluated_by', 'evaluated_on', 'encoded_on', 'updated_by', 'updated_on', 'time_endorsed', 'qc_type', 'formula_id', 'status_changed', 'operator', 'supervisor', 'bag_no', 'internal_lot']


## 2A\. Normalize all string columns

\(uppercase, remove whitespace\) — must run first, so validation only catches real format issues, not casing/spacing typos

In [3]:
# this block converts all col w/ string to a uniform format, all caps, no white spaces.
# this is to account and correct typos (eg white spaces, sometimes in lower caps etc)

def clean_column(column):
    """
    Remove all whitespace and convert text to uppercase.
    Only transforms non-null values — real blanks/NaN are never passed through
    .astype(str), so they can never become the literal text "NAN"/"NONE".
    """
    return column.where(
        column.isna(),
        column.astype(str).str.replace(r"\s+", "", regex=True).str.upper()
    )

str_col_to_clean = ['lot_number', 'product_code',
                'customer', 'original_lot', 'evaluated_by',
                'updated_by', 'updated_on', 'qc_type', 'formula_id',
                'operator', 'supervisor', 'bag_no', 'internal_lot']

for each_col in str_col_to_clean:
    qc_df[each_col] = clean_column(qc_df[each_col])

In [4]:
print (qc_df.head())

      id     lot_number product_code  \
0  14000         5187AO      IA1770E   
1  13999          0772Y   DE-B12696E   
2  13998         1155AO      KA5796E   
3  13997         5037AO      WA6365E   
4  13996  5034AO-5036AO      WA6365E   

                                     customer  status  \
0                         EVERBRIGHTNET&TWINE  Failed   
1                 H&EMANUFACTURINGCORPORATION  Passed   
2                 H&EMANUFACTURINGCORPORATION  Passed   
3  MEGAPLASTPACKAGING,INC.(FORMERLYPHILPLAST)  Passed   
4  MEGAPLASTPACKAGING,INC.(FORMERLYPHILPLAST)  Passed   

                                             remarks action   original_lot  \
0                blender1 of 4916AO-4917AO; & 5159AO    NaN         5159AO   
1                                                NaN    NaN          0772Y   
2  bluish; lab failed the remaining 5.42 kgs - sa...    NaN         1155AO   
3                                                NaN    NaN         5037AO   
4                         

## 2B\. Check ID for missing/gaps

> check missing prod ID in orig db

In [5]:
# ---- Step 2B: Validate QC file - Check IDs for missing/gaps ----
id_col = "id"

qc_ids = qc_df[id_col].dropna().astype(int)
id_min, id_max = qc_ids.min(), qc_ids.max()
full_range = set(range(id_min, id_max + 1))
missing_ids = sorted(full_range - set(qc_ids))
duplicate_ids = qc_ids[qc_ids.duplicated()].tolist()

print("=" * 60)
print("QC FILE - ID VALIDATION")
print("=" * 60)
print(f"ID range: {id_min} to {id_max}")
print(f"Total rows: {len(qc_ids)}")

if missing_ids:
    print(f"FLAGGED: {len(missing_ids)} missing ID(s) found.")
    # pd.DataFrame({"missing_id": missing_ids}).to_csv("flagged_missing_ids.csv", index=False)
    # print("Full list saved to flagged_missing_ids.csv for review.")
    # print(f"Missing ID sample: {missing_ids[:20]}")
else:
    print("No missing IDs found. Proceeding.")

if duplicate_ids:
    print(f"FLAGGED: {len(duplicate_ids)} duplicate ID(s) found.")
    print(f"Duplicate ID sample: {duplicate_ids[:20]}")
else:
    print("No duplicate IDs found.")

# ---- Step 2A (addendum): Split missing IDs by Aug 1, 2025 cutoff ----
"""
Dev noted missing IDs should not occur after Aug 1, 2025 — 
Implementation of new QC program any post-cutoff gap means
the record was excluded from the report export, not missing from the database.
It means that the record have value FALSE in is_active column in database.
"""

cutoff_id = 6493  # confirmed: first ID with encoded_on >= Aug 1, 2025

missing_before_cutoff = [i for i in missing_ids if i < cutoff_id]
missing_after_cutoff = [i for i in missing_ids if i >= cutoff_id]

print(f"\nMissing IDs before Aug 1, 2025 (id < {cutoff_id}): {len(missing_before_cutoff)}")
print(f"Missing IDs on/after Aug 1, 2025 (id >= {cutoff_id}): {len(missing_after_cutoff)}")

if missing_after_cutoff:
    print(f"FLAGGED: {len(missing_after_cutoff)} missing ID(s) found on & after the new program deployment.")
    # print(f"Sample: {missing_after_cutoff[:20]}")
else:
    print("No missing IDs on/after cutoff — consistent with dev's note.")

QC FILE - ID VALIDATION
ID range: 1 to 14000
Total rows: 12553
FLAGGED: 1447 missing ID(s) found.
No duplicate IDs found.

Missing IDs before Aug 1, 2025 (id < 6493): 574
Missing IDs on/after Aug 1, 2025 (id >= 6493): 873
FLAGGED: 873 missing ID(s) found on & after the new program deployment.


> format is 
DC \-
1\)  2 strings \(always\)
2\)  "\-" or none 
3\) ONE STRING 
3\) numbers \(infinite amount\)\- this should  be a given if you asked carefully
4\) 1 string 

if its valid and there is no "\-" add this to make it uniform

MB \-
1\)  2 strings \(always\)
2\)numbers \(infinite amount\)\- this should  be a given if you asked carefully
3\) 1 string 

to ask lab, since there exist the follwinng there there are more then 1 string in the end\. pls ask lab
     a\)   ID 12286: 'WA3827E\-I : is the std also the same code or its WA3827E?


     b\)    ID 9179: 'WA15190E1 :  is the std also the same code or its WA15190E1?

NOTE  \(IMPT\) : 
convert all to upper caps // remove white space = this should be std ALWAYS when validating or comparing 2 strings \. this eliminated one item so from 11 to 10 rows with unexpected product\_code format\.

## 2C\. Validate Product Codes

In [6]:
# ---- Step 2C: Validate QC file - Check product_code value if matched the gathered format ----
import re

# Confirmed formats (Convo A):
#   XX-X00000X  (5-digit, with client prefix)   e.g. DP-G13757E
#   XX-X0000X   (4-digit, with client prefix)   e.g. same structure, fewer digits
#   XX00000X or XX0000X (no prefix, plain code) e.g. RA16826E or VA4086E
PRODUCT_CODE_PATTERN = re.compile(
    r"^([A-Za-z]{2}-[A-Za-z]\d{4,5}[A-Za-z]|[A-Za-z]{2}\d{4,5}[A-Za-z])$"
)

def validate_product_code(value):
    if pd.isna(value):
        return False, "Blank/NaN product_code"
    value = str(value).strip()
    if not PRODUCT_CODE_PATTERN.match(value):
        return False, "Does not match confirmed format"
    return True, "OK"

print("=" * 60)
print("QC FILE - PRODUCT_CODE VALIDATION")
print("=" * 60)

product_code_issues = []

for idx, row in qc_df.iterrows():
    is_valid, reason = validate_product_code(row["product_code"])
    if not is_valid:
        product_code_issues.append({
            "row_index": idx,
            "id": int(row["id"]),
            "product_code": row["product_code"],
            "reason": reason
        })

if product_code_issues:
    print(f"FLAGGED: {len(product_code_issues)} row(s) with unexpected product_code format.")
    if len(product_code_issues) <= 20:
        for issue in product_code_issues:
            print(f"  ID {issue['id']}: '{issue['product_code']}'")
    else:
        pd.DataFrame(product_code_issues).to_csv("flagged_product_code.csv", index=False)
        print("Full list saved to flagged_product_code.csv for review.")
else:
    print("All product_code values passed validation.")

QC FILE - PRODUCT_CODE VALIDATION
FLAGGED: 10 row(s) with unexpected product_code format.
  ID 13180: 'DPG10045E'
  ID 12286: 'WA3827E-I'
  ID 9187: 'WA15190'
  ID 9179: 'WA15190E1'
  ID 5928: 'WA15190'
  ID 3844: 'D-I16082E'
  ID 3739: 'WA15123'
  ID 3046: 'GA1225'
  ID 3045: 'GA1225'
  ID 2188: 'RA121739E'


TEST BLOCK: TRY APPLYING CORRECTIONS\.

In [7]:
# ---- Step 2C (continued): Apply confirmed corrections and re-validate product_code ----

# New format confirmed with Jam:
#   XX0000X-X   (plain code + dash + 1 letter suffix) e.g. WA3827E-I — rare, confirmed with Ms. Jam
PRODUCT_CODE_PATTERN_UPDATED = re.compile(
    r"^([A-Za-z]{2}-[A-Za-z]\d{4,5}[A-Za-z]|[A-Za-z]{2}\d{4,5}[A-Za-z]|[A-Za-z]{2}\d{4,5}[A-Za-z]-[A-Za-z])$"
)

def validate_product_code_updated(value):
    if pd.isna(value):
        return False, "Blank/NaN product_code"
    value = str(value).strip()
    if not PRODUCT_CODE_PATTERN_UPDATED.match(value):
        return False, "Does not match confirmed format"
    return True, "OK"

# ---- Confirmed corrections from Ms. Jam for the 10 flagged rows ----
# Kept separate from the raw file — original value is preserved, correction is
# stored temporarily and only applied to the merged output later.
PRODUCT_CODE_CORRECTIONS = {
    13180: "DP-G10045E",
    9187: "WA15190E",
    9179: "WA15190E",   # confirmed typo, same as WA15190E
    5928: "WA15190E",
    3844: "DV-I16082E",
    3739: "WA15123E",
    3046: "GA1225E",
    3045: "GA1225E",
    2188: "RA12739E",
    # 12286 (WA3827E-I) is not a typo — it's a newly confirmed valid format, no correction needed
}

qc_df["product_code_corrected"] = qc_df.apply(
    lambda row: PRODUCT_CODE_CORRECTIONS.get(int(row["id"]), row["product_code"]),
    axis=1
)

print("=" * 60)
print("QC FILE - PRODUCT_CODE VALIDATION (after corrections)")
print("=" * 60)

product_code_issues_updated = []

for idx, row in qc_df.iterrows():
    is_valid, reason = validate_product_code_updated(row["product_code_corrected"])
    if not is_valid:
        product_code_issues_updated.append({
            "row_index": idx,
            "id": int(row["id"]),
            "product_code": row["product_code"],
            "product_code_corrected": row["product_code_corrected"],
            "reason": reason
        })

if product_code_issues_updated:
    print(f"FLAGGED: {len(product_code_issues_updated)} row(s) with unexpected product_code format.")
    if len(product_code_issues_updated) <= 20:
        for issue in product_code_issues_updated:
            print(f"  ID {issue['id']}: '{issue['product_code']}'")
    else:
        pd.DataFrame(product_code_issues_updated).to_csv("flagged_product_code.csv", index=False)
        print("Full list saved to flagged_product_code.csv for review.")
else:
    print("All product_code values passed validation after corrections.")

QC FILE - PRODUCT_CODE VALIDATION (after corrections)
All product_code values passed validation after corrections.


LENGTH CHECK AFTER APPLYING TEST CORRECTIONS IN PRODUCT CODE:

In [8]:
def is_covered_format(value):
    value = str(value).strip()

    if re.match(r"^[A-Za-z]{2}-[A-Za-z]\d{4,5}[A-Za-z]$", value):
        return True
    if re.match(r"^[A-Za-z]{2}\d{4,5}[A-Za-z]-[A-Za-z]$", value):
        return True
    if re.match(r"^[A-Za-z]{2}\d{4,5}[A-Za-z]$", value):
        return True

    return False

qc_df["id"] = qc_df["id"].astype(int)
qc_df["product_code_is_covered"] = qc_df["product_code_corrected"].apply(is_covered_format)

uncovered = qc_df[qc_df["product_code_is_covered"] == False]

print(f"Total rows: {len(qc_df)}")
print(f"Covered by known format: {(qc_df['product_code_is_covered'] == True).sum()}")
print(f"NOT covered by any known format: {len(uncovered)}")

if len(uncovered) > 0:
    print("\nUncovered product_code values:")
    print(uncovered[["id", "product_code_corrected"]].to_string(index=False))
else:
    print("\nAll product_code values are covered by a known format.")

Total rows: 12553
Covered by known format: 12553
NOT covered by any known format: 0

All product_code values are covered by a known format.


## 2D\. Validate Lot Number

Confirm all real formats present, narrowed down step by step: single lot → range → parenthesis \(bag\-in\-parens / internal\-lot\-in\-parens\) →        whatever remains is flagged

In [9]:
# check lot format

# put all lot data in a list
lot_number_list = qc_df['lot_number'].tolist()

### \- 2D Validation Check 1: Exclude valid lot numbers in the list\.

In [10]:
# CHECK THE POSSIBLE VALID FORMATS of the lot number column

#(1) remove the ones where format is SSSSNN or SSSSN as it is valid 
pattern = re.compile(r"^\d{4}[A-Za-z]{1,2}$")
invalid_lot_number_list = [item for item in lot_number_list if not pattern.match(str(item))]


# NOTE
# lot_number_list len = 12,339
# invalid_lot_number_list len  = 3,879


### \- 2D Validation Check 2: List valid lot number ranges\.

In [11]:
#(2) remove those w/ this format SSSSNN-SSSSNN and SSSSN-SSSSN where , 
# all 3 conditions below must be met 
#    a) all number must have "- " in b/w
#    b) b4 and after must be in the format listed above and same on each side
#    c) first number < 2nd number 
# as they are all VALID 



'''
for (a) and (b)
pattern is : 
    There is exactly one -
    Left side is either: NNNNS (4 digits + 1 letter), or  NNNNSS (4 digits + 2 letters)
    Right side is the same format as the left side (either both 1 letter or both 2 letters).
'''
pattern = re.compile(
    r'^(?:\d{4}[A-Za-z]-\d{4}[A-Za-z]|\d{4}[A-Za-z]{2}-\d{4}[A-Za-z]{2})$'
)

# temp_list are the VALID lot number 
temp_list = [item for item in invalid_lot_number_list if pattern.match(str(item))]




In [12]:
print (temp_list)
print (len(temp_list))

['5034AO-5036AO', '5168AO-5171AO', '0764Y-0765Y', '0762Y-0763Y', '5081AO-5086AO', '5068AO-5080AO', '4973AO-4975AO', '5027AO-5032AO', '0766Y-0767Y', '4969AO-4972AO', '4963AO-4968AO', '5019AO-5021AO', '5048AO-5059AO', '4842AO-4843AO', '4888AO-4917AO', '4795AO-4803AO', '4845AO-4848AO', '4852AO-4861AO', '4806AO-4814AO', '4831AO-4836AO', '4827AO-4829AO', '4863AO-4885AO', '4816AO-4824AO', '4792AO-4793AO', '4756AO-4758AO', '0740Y-0748Y', '4526AO-4529AO', '4760AO-4764AO', '4785AO-4787AO', '4781AO-4783AO', '4520AO-4525AO', '4550AM-4551AM', '0728Y-0737Y', '4711AO-4726AO', '4769AO-4774AO', '4698AO-4706AO', '5651AN-5653AN', '0721Y-0727Y', '4509AO-4519AO', '4568AO-4572AO', '4545AO-4548AO', '4553AO-4558AO', '4530AO-4543AO', '4559AO-4562AO', '4603AO-4612AO', '4489AO-4501AO', '6642AL-6643AL', '5088AL-5091AL', '7267AM-7270AM', '0280AN-0281AN', '3251AN-3261AN', '8536AN-8552AN', '8777AN-8784AN', '4486AO-4487AO', '4575AO-4581AO', '3959AO-3961AO', '4595AO-4596AO', '4586AO-4593AO', '4565AO-4566AO', '0691Y-0

### \- 2D Validation Check 3: Confirm valid lot number range order

In [13]:
#then (c)... must be in consecutive order or 1st<2nd , eg 1000AB-1200AB or 1000AA-1000AB,  
# if not then remove them as they are invalid


def remove_invalid_ranges(my_list):
    pattern = re.compile(r"^(\d{4})([A-Za-z]{1,2})\s*-\s*(\d{4})([A-Za-z]{1,2})$")

    cleaned_list = []

    for item in my_list:
        match = pattern.match(str(item).strip())

        # If format does not match, keep it
        if not match:
            cleaned_list.append(item)
            continue

        nnnn = int(match.group(1))
        left_letters = match.group(2).upper()

        NNNN = int(match.group(3))
        right_letters = match.group(4).upper()

        # If left and right letter length are not the same, keep it
        if len(left_letters) != len(right_letters):
            cleaned_list.append(item)
            continue

        remove_item = False

        # Case 1: 4 digits + 1 letter
        if len(left_letters) == 1:
            s = left_letters
            S = right_letters

            if s == S:
                if NNNN <= nnnn:
                    remove_item = True
            else:
                if s > S:
                    remove_item = True

        # Case 2: 4 digits + 2 letters
        elif len(left_letters) == 2:
            ss = left_letters
            SS = right_letters

            t = ss[-1]
            T = SS[-1]

            if ss == SS:
                if NNNN <= nnnn:
                    remove_item = True
            else:
                if t > T:
                    remove_item = True

        if not remove_item:
            cleaned_list.append(item)

    return cleaned_list



temp_list = remove_invalid_ranges(temp_list)

# seems nothing was remove and ranges are valid

# remove items in temp_list from invalid_lot_number_list as temp_list are all valid lots
invalid_lot_number_list = [
    x for x in invalid_lot_number_list
    if x not in temp_list
]

# invalid_Lot_number_list len now at 1000

In [14]:
print (invalid_lot_number_list)
print (len(invalid_lot_number_list))

['6124AM(6127AM)', '5192AM(1-40)', '2406AM(1-16)', '9345AM-9348AM`', '8670AM-7671AM', '7200AM(10)', '7200AM(16-18)', '7200AM(1-9)', '7745AM(1-2)', '7495AM(1-26)', '7594AM(38-39)', '7293AM(1-20)', '7461AM(35-37)', '7461AM(1-34)', '7441AM(1-40)', '7273AM(39-40)', '7636AM(1-4)', '7562AM(1-3)', '7631AM(41)', '7273AM(9-38)', '7273AM(1-8)', '7199AM(1-2)', '7494AM(20)', '7323AM(15-18)', '7323AM(1-14)', '7303AM(37-40)', '7303AM(19-36)', '7303AM(1-18)', '7190AM-7192AM(15-19)', '7196AM(7198AM)', '7196AM(7197AM)', '7149AM(25-27)', '7149AM(1-24)', '7129AM(27-40)', '7200AM(11-15)', '7129AM(1-26)', '7043AM(1-2)', '7107AM(1-4)', '7109AM(1-40)', '6743AM(13-14)', '7023AM(7-40)', '6921AM(4)', '6921AM(1-3)', '7023AM(1-6)', '7003AM(1-40)', '6964AM(1-40)', '7085AM(39-40)', '7085AM(1-38)', '6944AM(13-40)', '7065AM(1-40)', '7045AM(15-40)', '6944AM(1-12)', '6924AM(19-40)', '7045AM(1-14)', '6924AM(1-18)', '6890AM(15)', '6857AM(27)', '6755AM(13)', '6676AM(21)', '6675AM(20)', '6803AM(11)', '6766AM(16)', '6605AM(

### \- 2D Validation Check 4: Remove valid parenthesis\-format bag numbers \(single bag\)

In [15]:
# Format: 1234AB(#) or 1234A(#) — valid base lot, single bag number in parentheses
# e.g. 7200AM(10), 6921AM(4), 3067AM(2)

pattern = re.compile(
    r'^\d{4}[A-Za-z]{1,2}\(\d+\)$'
)

# temp_list_bag_single are the VALID entries for this pass
temp_list_bag_single = [item for item in invalid_lot_number_list if pattern.match(str(item).strip())]

print(temp_list_bag_single)
print(len(temp_list_bag_single))


['7200AM(10)', '7631AM(41)', '7494AM(20)', '6921AM(4)', '6890AM(15)', '6857AM(27)', '6755AM(13)', '6676AM(21)', '6675AM(20)', '6803AM(11)', '6766AM(16)', '6605AM(10)', '6474AM(9)', '6557AM(31)', '6616AM(21)', '6588AM(21)', '6342AM(32)', '6341AM(31)', '6521AM(31)', '6520AM(30)', '6350AM(21)', '6401AM(38)', '6123AM(2)', '5853AM(15)', '6193AM(1)', '5873AM(10)', '5821AM(41)', '5583AM(20)', '5521AM(21)', '5111AM(41)', '5140AM(29)', '5161AM(21)', '5296AM(36)', '5071AM(39)', '5016AM(9)', '4941AM(39)', '4990AM(21)', '5009AM(11)', '4879AM(11)', '4745AM(1)', '4703AM(20)', '4833AM(36)', '4596AM(41)', '4595AM(40)', '4676AM(1)', '4221AM(39)', '4364AM(20)', '4363AM(19)', '3896AM(7)', '4043AM(24)', '3809AM(30)', '3794AM(20)', '3475AM(21)', '3348AM(1)', '3280AM(10)', '3054AM(21)', '3059AM(20)', '3067AM(2)', '3004AM(21)', '2982AM(17)', '3005AM(10)', '7469AL(4)', '2897AM(11)', '2896AM(10)', '2866AM(33)', '2733AM(11)', '2645AM(17)', '2687AM(35)', '2571AM(9)', '2570AM(8)', '2508AM(5)', '1794AM(13)', '2241

In [16]:
# Remove these confirmed-valid entries from invalid_lot_number_list
invalid_lot_number_list = [
    x for x in invalid_lot_number_list
    if x not in temp_list_bag_single
]

print(f"Remaining invalid after Check 4: {len(invalid_lot_number_list)}")

Remaining invalid after Check 4: 811


### \- 2D Validation Check 5: Remove valid parenthesis\-format bag ranges

In [17]:
# ---- Step 2D Pass 5:  ----
# Format: 1234AB(#a-#b) where a < b — valid base lot, bag range in parentheses
# e.g. 7200AM(1-9), 7461AM(35-37), 7594AM(38-39)

pattern = re.compile(
    r'^\d{4}[A-Za-z]{1,2}\((\d+)-(\d+)\)$'
)

temp_list_bag_range = []

for item in invalid_lot_number_list:
    match = pattern.match(str(item).strip())
    if match:
        a, b = int(match.group(1)), int(match.group(2))
        if a < b:
            temp_list_bag_range.append(item)

print(temp_list_bag_range)
print(len(temp_list_bag_range))

['5192AM(1-40)', '2406AM(1-16)', '7200AM(16-18)', '7200AM(1-9)', '7745AM(1-2)', '7495AM(1-26)', '7594AM(38-39)', '7293AM(1-20)', '7461AM(35-37)', '7461AM(1-34)', '7441AM(1-40)', '7273AM(39-40)', '7636AM(1-4)', '7562AM(1-3)', '7273AM(9-38)', '7273AM(1-8)', '7199AM(1-2)', '7323AM(15-18)', '7323AM(1-14)', '7303AM(37-40)', '7303AM(19-36)', '7303AM(1-18)', '7149AM(25-27)', '7149AM(1-24)', '7129AM(27-40)', '7200AM(11-15)', '7129AM(1-26)', '7043AM(1-2)', '7107AM(1-4)', '7109AM(1-40)', '6743AM(13-14)', '7023AM(7-40)', '6921AM(1-3)', '7023AM(1-6)', '7003AM(1-40)', '6964AM(1-40)', '7085AM(39-40)', '7085AM(1-38)', '6944AM(13-40)', '7065AM(1-40)', '7045AM(15-40)', '6944AM(1-12)', '6924AM(19-40)', '7045AM(1-14)', '6924AM(1-18)', '6473AM(7-8)', '6617AM(1-5)', '6402AM(39-40)', '5853AM(16-18)', '5767AM(23-24)', '5767AM(15-22)', '5853AM(1-14)', '5767AM(1-14)', '5747AM(31-40)', '5833AM(1-40)', '5747AM(1-30)', '5831AM(1-4)', '6263AM(1-9)', '5629AM(38-40)', '5629AM(1-37)', '5727AM(11-40)', '5609AM(9-40)',

In [18]:
# Remove these confirmed-valid entries from invalid_lot_number_list
invalid_lot_number_list = [
    x for x in invalid_lot_number_list
    if x not in temp_list_bag_range
]

print(f"Remaining invalid after Check 5: {len(invalid_lot_number_list)}")

Remaining invalid after Check 5: 183


### \- 2D Validation Check 6: Remove valid parenthesis\-format internal lot numbers

In [19]:
# ---- Step 2D Pass 6:  ----
# Format: 1234AB(5678CD) — valid base lot, real internal lot (letters + numbers) in parentheses
# e.g. 6124AM(6127AM), 6240AM(6249AM), 2904AM(2914AM)

pattern = re.compile(
    r'^\d{4}[A-Za-z]{1,2}\((\d{4}[A-Za-z]{1,2})\)$'
)

temp_list_internal_lot = [item for item in invalid_lot_number_list if pattern.match(str(item).strip())]

print(temp_list_internal_lot)
print(len(temp_list_internal_lot))

# Remove these confirmed-valid entries from invalid_lot_number_list
invalid_lot_number_list = [
    x for x in invalid_lot_number_list
    if x not in temp_list_internal_lot
]

print(f"Remaining invalid after Check 6: {len(invalid_lot_number_list)}")

['6124AM(6127AM)', '7196AM(7198AM)', '7196AM(7197AM)', '6344AM(6345AM)', '6240AM(6249AM)', '6240AM(6248AM)', '6240AM(6247AM)', '6240AM(6246AM)', '6240AM(6245AM)', '6240AM(6244AM)', '6240AM(6243AM)', '6240AM(6242AM)', '6240AM(6241AM)', '6124AM(6138AM)', '6124AM(6137AM)', '6124AM(6136AM)', '6124AM(6135AM)', '6124AM(6134AM)', '6124AM(6133AM)', '6124AM(6132AM)', '6124AM(6131AM)', '6124AM(6130AM)', '6124AM(6129AM)', '6124AM(6128AM)', '5793AM(5797AM)', '6124AM(6126AM)', '6124AM(6125AM)', '5899AM(5901AM)', '5899AM(5900AM)', '5793AM(5794AM)', '5793AM(5796AM)', '5793AM(5795AM)', '4994AM(4995AM)', '4400AM(4409AM)', '4400AM(4408AM)', '4400AM(4407AM)', '4400AM(4406AM)', '4400AM(4405AM)', '4400AM(4404AM)', '4400AM(4403AM)', '4400AM(4402AM)', '4400AM(4401AM)', '2820AM(2822AM)', '2904AM(2914AM)', '2904AM(2913AM)', '2904AM(2912AM)', '2904AM(2911AM)', '2904AM(2910AM)', '2904AM(2909AM)', '2904AM(2908AM)', '2904AM(2907AM)', '2904AM(2906AM)', '2904AM(2905AM)', '2820AM(2825AM)', '2820AM(2824AM)', '2820AM(2

In [20]:
print(invalid_lot_number_list)
print(len(invalid_lot_number_list))

['9345AM-9348AM`', '8670AM-7671AM', '7190AM-7192AM(15-19)', '0764AM-0365AM', '86342AL(8344AL)', '8761AL-8762AL(39-40)', '8086AL-8087AL(40-41)', '7839AL-7840AL(9-11)', '7786AL-7788AL(17-24)', '7306AL-7307AL(40-41)', '6965AL`', '5466AL(5473AL(', '4506AL--4522AL', '4034AAL-4039AL']
14


### \- 2D Validation Check 7: Check invalid Lot Numbers

In [21]:
print(invalid_lot_number_list)
print(len(invalid_lot_number_list))

['9345AM-9348AM`', '8670AM-7671AM', '7190AM-7192AM(15-19)', '0764AM-0365AM', '86342AL(8344AL)', '8761AL-8762AL(39-40)', '8086AL-8087AL(40-41)', '7839AL-7840AL(9-11)', '7786AL-7788AL(17-24)', '7306AL-7307AL(40-41)', '6965AL`', '5466AL(5473AL(', '4506AL--4522AL', '4034AAL-4039AL']
14


TEST BLOCK: TRY APPLYING CORRECTIONS FOR TYPOS\.

In [22]:
# ---- Step 2D: Apply confirmed corrections for lot_number anomalies ----
# Kept separate from the raw file — original value is preserved, correction is
# stored temporarily and only applied to the merged output later.

LOT_NUMBER_CORRECTIONS = {
    "33478AO"         : "3478AO",
    "9345AM-9348AM`"  : "9345AM-9348AM",
    "8670AM-7671AM"   : "8670AM-8671AM",
    "4550-4551AM"     : "4550AM-4551AM",
    "0764AM-0365AM"   : "0364AM-0365AM",
    "86342AL(8344AL)" : "8342AL(8344AL)",
    "6965AL`"         : "6965AL",
    "5466AL(5473AL("  : "5466AL(5473AL)",
    "4506AL--4522AL"  : "4506AL-4522AL",
    "4034AAL-4039AL"  : "4034AL-4039AL",
}

qc_df["lot_number_corrected"] = qc_df["lot_number"].apply(
    lambda val: LOT_NUMBER_CORRECTIONS.get(str(val).strip(), val)
)

corrected_count = (qc_df["lot_number"] != qc_df["lot_number_corrected"]).sum()
print(f"Applied {corrected_count} lot_number correction(s).")
print(qc_df.loc[qc_df["lot_number"] != qc_df["lot_number_corrected"],
                 ["id", "lot_number", "lot_number_corrected"]])

Applied 8 lot_number correction(s).
         id       lot_number lot_number_corrected
6028   7217   9345AM-9348AM`        9345AM-9348AM
6309   6886    8670AM-7671AM        8670AM-8671AM
9094   3810    0764AM-0365AM        0364AM-0365AM
9635   3224  86342AL(8344AL)       8342AL(8344AL)
10302  2505          6965AL`               6965AL
11004  1720   5466AL(5473AL(       5466AL(5473AL)
11519  1146   4506AL--4522AL        4506AL-4522AL
11633  1028   4034AAL-4039AL        4034AL-4039AL


In [23]:
# ---- Step 2D: Update invalid_lot_number_list to remove corrected entries ----
# Removes the 10 confirmed-and-corrected values from invalid_lot_number_list,
# since they are no longer actually invalid — just needed a fix.

invalid_lot_number_list = [
    item for item in invalid_lot_number_list
    if str(item).strip() not in LOT_NUMBER_CORRECTIONS
]

print(invalid_lot_number_list)
print(len(invalid_lot_number_list))

['7190AM-7192AM(15-19)', '8761AL-8762AL(39-40)', '8086AL-8087AL(40-41)', '7839AL-7840AL(9-11)', '7786AL-7788AL(17-24)', '7306AL-7307AL(40-41)']
6


### \- 2D Validation Check 8: Remove valid lot number range \+ parenthesis\-format bag ranges

In [24]:
# ---- Step 2D: Identify valid Lot Ranged + Parenthesised Ranged Bag (equal count) ----
# Format: 0000AA-0001AA(5-6) — valid ONLY if the count of lots in the range
# matches the count of bags in the range. Each lot pairs to one bag in sequential order.
# ex: 0000AA(5), 0001AA(6)

pattern = re.compile(
    r'^(\d{4})([A-Za-z]{1,2})-(\d{4})([A-Za-z]{1,2})\((\d+)-(\d+)\)$'
)

resolved_equal_originals = []

for item in invalid_lot_number_list:
    match = pattern.match(str(item).strip())
    if not match:
        continue

    start_num, start_letters, end_num, end_letters, bag_start, bag_end = match.groups()

    # Letters on both sides of the lot range must match (same series)
    if start_letters.upper() != end_letters.upper():
        continue

    lot_count = int(end_num) - int(start_num) + 1
    bag_count = int(bag_end) - int(bag_start) + 1

    # Valid only if lot count matches bag count, and both are in ascending order
    if lot_count == bag_count and int(end_num) > int(start_num) and int(bag_end) > int(bag_start):
        resolved_equal_originals.append(item)

print(resolved_equal_originals)
print(len(resolved_equal_originals))

['8761AL-8762AL(39-40)', '8086AL-8087AL(40-41)', '7306AL-7307AL(40-41)']
3


TEST BLOCK: TRY APPLYING CORRECTIONS FOR 3 RANGE \+ BAG RANGE \(ONE LOT = ONE BAG\)

In [25]:
# ---- Step 2D: Split each confirmed equal-count combo entry into individual lot(bag) pairs ----

temp_list_lot_range_bag_range = []

for item in resolved_equal_originals:
    match = pattern.match(str(item).strip())
    start_num, start_letters, end_num, end_letters, bag_start, bag_end = match.groups()
    num_len = len(start_num)

    lot_numbers = [f"{str(n).zfill(num_len)}{start_letters.upper()}" for n in range(int(start_num), int(end_num) + 1)]
    bag_numbers = list(range(int(bag_start), int(bag_end) + 1))

    for lot, bag in zip(lot_numbers, bag_numbers):
        temp_list_lot_range_bag_range.append(f"{lot}({bag})")

print(temp_list_lot_range_bag_range)
print(len(temp_list_lot_range_bag_range))

invalid_lot_number_list = [
    x for x in invalid_lot_number_list
    if x not in resolved_equal_originals
]

print(f"Remaining invalid after Check 8: {len(invalid_lot_number_list)}")

['8761AL(39)', '8762AL(40)', '8086AL(40)', '8087AL(41)', '7306AL(40)', '7307AL(41)']
6
Remaining invalid after Check 8: 3


In [26]:
invalid_lot_number_list = [
    x for x in invalid_lot_number_list
    if x not in temp_list_lot_range_bag_range
]

print(invalid_lot_number_list)
print(len(invalid_lot_number_list))

['7190AM-7192AM(15-19)', '7839AL-7840AL(9-11)', '7786AL-7788AL(17-24)']
3


TEST BLOCK: TRY APPLYING CORRECTIONS FOR REMAINING INVALID LOT NUMBER

In [27]:
# ---- Manual corrections for uneven Lot Range + Bag Range mismatches ----
# These 3 entries have a lot range count that does NOT match the bag range count,
# so they can't auto-split evenly. Corrected manually and split per-lot as confirmed.

# (3) has a raw data error in the bag range itself — fix that first before splitting.
RAW_VALUE_CORRECTIONS = {
    "7786AL-7788AL(17-24)": "7786AL-7788AL(15-24)",
}

# Confirmed manual split — each original range entry maps to its corrected individual lot+bag rows.
UNEVEN_RANGE_BAG_SPLITS = {
    "7190AM-7192AM(15-19)": ["7190AM(15-16)", "7191AM(17-18)", "7192AM(19)"],
    "7839AL-7840AL(9-11)": ["7839AL(9-10)", "7840AL(11)"],
    "7786AL-7788AL(15-24)": ["7786AL(15-16)", "7787AL(17-20)", "7788AL(21-24)"],
}

temp_list_uneven_split = []
resolved_originals = []

for original_value in list(invalid_lot_number_list):
    corrected_raw = RAW_VALUE_CORRECTIONS.get(original_value, original_value)

    if corrected_raw in UNEVEN_RANGE_BAG_SPLITS:
        temp_list_uneven_split.extend(UNEVEN_RANGE_BAG_SPLITS[corrected_raw])
        resolved_originals.append(original_value)

print(temp_list_uneven_split)
print(len(temp_list_uneven_split))

['7190AM(15-16)', '7191AM(17-18)', '7192AM(19)', '7839AL(9-10)', '7840AL(11)', '7786AL(15-16)', '7787AL(17-20)', '7788AL(21-24)']
8


In [28]:
# Remove these resolved entries from invalid_lot_number_list
invalid_lot_number_list = [
    x for x in invalid_lot_number_list
    if x not in resolved_originals
]

print(invalid_lot_number_list)
print(len(invalid_lot_number_list))

[]
0


## 2E\. Classify QC lot\_number initial formats

In [29]:
def classify_qc_lot(value):
    value = str(value).strip()

    if value == "" or value.lower() == "nan":
        return "blank"
    if "(" in value and ")" in value:
        return "parenthesis"
    if "-" in value:
        return "range"
    return "single"

qc_df["lot_format"] = qc_df["lot_number_corrected"].apply(classify_qc_lot)

print("Count of each lot_number format:")
print(qc_df["lot_format"].value_counts())
print(f"\nTotal classified rows: {qc_df['lot_format'].value_counts().sum()}")
print(f"Total raw QC file rows (after dropping blank spacer rows): {len(qc_df)}")

Count of each lot_number format:
lot_format
single         8639
range          2922
parenthesis     992
Name: count, dtype: int64

Total classified rows: 12553
Total raw QC file rows (after dropping blank spacer rows): 12553


## 2F\. Expand range\-format lot numbers

In [30]:
# ---- Step 2G: Expand range-based QC lot_number values into individual lots ----

# Lot number format 0000XX - Split into value (0000 and XX)
def split_lot_parts(lot): 
    match = re.match(r"^(\d+)([A-Za-z]+)$", lot.strip())
    if not match:
        return None, None, None
    number_str, letters = match.group(1), match.group(2)
    return number_str, letters, len(number_str)

# Get the next letter combo in the lot number string, like AM turning into AN.
def next_letters(letters): 
    letters = list(letters.upper())
    i = len(letters) - 1
    while i >= 0:
        if letters[i] != 'Z':
            letters[i] = chr(ord(letters[i]) + 1)
            break
        else:
            letters[i] = 'A'
            i -= 1
    return "".join(letters)

# Split the range: eg 6222AM-6224AM -> 6222AM, 6223AM, 6224AM
def expand_range(start_lot, end_lot): 
    start_num_str, start_letters, num_len = split_lot_parts(start_lot)
    end_num_str, end_letters, _ = split_lot_parts(end_lot)

    if start_num_str is None or end_num_str is None:
        return None

    result = []
    current_num = int(start_num_str)
    current_letters = start_letters
    max_value = 10 ** num_len - 1

    safety_counter = 0
    max_iterations = 5000

    while True:
        current_lot = f"{str(current_num).zfill(num_len)}{current_letters}"
        result.append(current_lot)

        if current_lot == end_lot.strip().upper():
            break

        current_num += 1
        if current_num > max_value:
            current_num = 1
            current_letters = next_letters(current_letters)

        safety_counter += 1
        if safety_counter > max_iterations:
            print(f"WARNING: range {start_lot}-{end_lot} exceeded safety limit, stopped expanding.")
            break

    return result

# ---- Apply range expansion to all range-format rows ----
range_rows = qc_df[qc_df["lot_format"] == "range"].copy()

expanded_records = []
failed_ranges = []

for idx, row in range_rows.iterrows():
    parts = row["lot_number_corrected"].split("-")
    if len(parts) != 2:
        failed_ranges.append(row["lot_number_corrected"])
        continue

    start_lot, end_lot = parts[0].strip(), parts[1].strip()
    expanded_lots = expand_range(start_lot, end_lot)

    if expanded_lots is None:
        failed_ranges.append(row["lot_number_corrected"])
        continue

    for lot in expanded_lots:
        new_row = row.copy()
        new_row["expanded_lot_number"] = lot
        expanded_records.append(new_row)

expanded_df = pd.DataFrame(expanded_records)

print(f"Total range rows: {len(range_rows)}")
print(f"Total individual lots after expansion: {len(expanded_df)}")
print(f"Ranges that failed to parse: {len(failed_ranges)}")
if failed_ranges:
    # print("Failed range samples:", failed_ranges[:10])
    print("Failed range samples:", failed_ranges)

Total range rows: 2922
Total individual lots after expansion: 20288
Ranges that failed to parse: 0


## 2G\. Lot number format breakdown

In [31]:
# ---- Step 2H Part 1: Build ALL_COMBO_CORRECTIONS from earlier resolved lists ----
# Reuses what earlier passes already computed, instead of hardcoding a second copy.

ALL_COMBO_CORRECTIONS = {}

# From the uneven manual splits
for original_value in resolved_originals:  # e.g. 7190AM-7192AM(15-19)
    corrected_raw = RAW_VALUE_CORRECTIONS.get(original_value, original_value)
    ALL_COMBO_CORRECTIONS[original_value] = UNEVEN_RANGE_BAG_SPLITS[corrected_raw]

# From the equal-count auto splits
for original_value in resolved_equal_originals:  # e.g. 8761AL-8762AL(39-40)
    match = pattern.match(original_value)
    start_num, start_letters, end_num, _, bag_start, bag_end = match.groups()
    num_len = len(start_num)
    lot_numbers = [f"{str(n).zfill(num_len)}{start_letters.upper()}" for n in range(int(start_num), int(end_num) + 1)]
    bag_numbers = list(range(int(bag_start), int(bag_end) + 1))
    ALL_COMBO_CORRECTIONS[original_value] = [f"{lot}({bag})" for lot, bag in zip(lot_numbers, bag_numbers)]

print(ALL_COMBO_CORRECTIONS)
print(f"Total combo entries built: {len(ALL_COMBO_CORRECTIONS)}")

{'7190AM-7192AM(15-19)': ['7190AM(15-16)', '7191AM(17-18)', '7192AM(19)'], '7839AL-7840AL(9-11)': ['7839AL(9-10)', '7840AL(11)'], '7786AL-7788AL(17-24)': ['7786AL(15-16)', '7787AL(17-20)', '7788AL(21-24)'], '8761AL-8762AL(39-40)': ['8761AL(39)', '8762AL(40)'], '8086AL-8087AL(40-41)': ['8086AL(40)', '8087AL(41)'], '7306AL-7307AL(40-41)': ['7306AL(40)', '7307AL(41)']}
Total combo entries built: 6


In [32]:
# ---- Step 2H Part 2: Expand combo rows into qc_df, then classify all lot_number formats ----

rows_to_expand = qc_df[qc_df["lot_number_corrected"].isin(ALL_COMBO_CORRECTIONS.keys())]
expanded_combo_records = []

for idx, row in rows_to_expand.iterrows():
    for new_lot in ALL_COMBO_CORRECTIONS[row["lot_number_corrected"]]:
        new_row = row.copy()
        new_row["lot_number_corrected"] = new_lot
        expanded_combo_records.append(new_row)

qc_df = qc_df[~qc_df["lot_number_corrected"].isin(ALL_COMBO_CORRECTIONS.keys())]
qc_df = pd.concat([qc_df, pd.DataFrame(expanded_combo_records)], ignore_index=True)

print(f"Expanded {len(rows_to_expand)} combo row(s) into {len(expanded_combo_records)} individual row(s).")
print(f"qc_df total rows now: {len(qc_df)}")

# ---- Classify every lot_number format ----
BASE_LOT = r"\d{4}[A-Za-z]{1,2}"

PATTERNS = {
    "single":             re.compile(rf"^{BASE_LOT}$"),
    "range":              re.compile(rf"^{BASE_LOT}-{BASE_LOT}$"),
    "bag_single_paren":   re.compile(rf"^{BASE_LOT}\(\d+\)$"),
    "bag_range_paren":    re.compile(rf"^{BASE_LOT}\(\d+-\d+\)$"),
    "internal_lot_paren": re.compile(rf"^{BASE_LOT}\({BASE_LOT}\)$"),
}

format_buckets = {name: [] for name in PATTERNS}
unclassified = []

for idx, row in qc_df.iterrows():
    value = str(row["lot_number_corrected"]).strip()
    entry = {"id": int(row["id"]), "product_code": row["product_code_corrected"], "lot_number": value}

    matched_any = False
    for name, pattern_re in PATTERNS.items():
        if pattern_re.match(value):
            format_buckets[name].append(entry)
            matched_any = True
            break

    if not matched_any:
        unclassified.append(entry)

print("=" * 60)

for name, entries in format_buckets.items():
    print(f"{name}: {len(entries)} row(s)")

print(f"\nUnclassified (no known format matched): {len(unclassified)} row(s)")
if unclassified:
    for r in unclassified:
        print(f"  ID {r['id']}: product_code='{r['product_code']}', lot_number='{r['lot_number']}'")
else:
    print("All lot_number values are now covered by a known format.")

Expanded 6 combo row(s) into 14 individual row(s).
qc_df total rows now: 12561
single: 8639 row(s)
range: 2922 row(s)
bag_single_paren: 195 row(s)
bag_range_paren: 634 row(s)
internal_lot_paren: 171 row(s)

Unclassified (no known format matched): 0 row(s)
All lot_number values are now covered by a known format.


In [33]:
# ---- Step 2H (addendum): Explain the row count change ----
# qc_df grew from 12,339 to 12,347 rows after this step.
# This is expected — not a data error. Here's why:

print("=" * 60)
print("ROW COUNT CHANGE EXPLANATION")
print("=" * 60)
print(f"Rows before this step  : 12339")
print(f"Rows after this step   : {len(qc_df)}")
print(f"Difference             : {len(qc_df) - 12339}")

print("\nReason: 6 QC rows had a 'lot range + bag range' combo format")
print("(e.g. 8761AL-8762AL(39-40)), which is not a single real lot — it actually")
print("represents multiple individual lots, each with its own bag number.")
print("This step expanded each of those 6 rows into several individual rows,")
print("one per real lot(bag) pair, so each one can be matched correctly later.")

print("\nBreakdown of the expansion:")
for original, expanded_list in ALL_COMBO_CORRECTIONS.items():
    print(f"  '{original}'  ->  {len(expanded_list)} row(s): {expanded_list}")

total_expanded = sum(len(v) for v in ALL_COMBO_CORRECTIONS.values())
print(f"\n6 original rows expanded into {total_expanded} individual rows.")
print(f"Net change: {total_expanded} - 6 = {total_expanded - 6} additional rows.")

ROW COUNT CHANGE EXPLANATION
Rows before this step  : 12339
Rows after this step   : 12561
Difference             : 222

Reason: 6 QC rows had a 'lot range + bag range' combo format
(e.g. 8761AL-8762AL(39-40)), which is not a single real lot — it actually
represents multiple individual lots, each with its own bag number.
This step expanded each of those 6 rows into several individual rows,
one per real lot(bag) pair, so each one can be matched correctly later.

Breakdown of the expansion:
  '7190AM-7192AM(15-19)'  ->  3 row(s): ['7190AM(15-16)', '7191AM(17-18)', '7192AM(19)']
  '7839AL-7840AL(9-11)'  ->  2 row(s): ['7839AL(9-10)', '7840AL(11)']
  '7786AL-7788AL(17-24)'  ->  3 row(s): ['7786AL(15-16)', '7787AL(17-20)', '7788AL(21-24)']
  '8761AL-8762AL(39-40)'  ->  2 row(s): ['8761AL(39)', '8762AL(40)']
  '8086AL-8087AL(40-41)'  ->  2 row(s): ['8086AL(40)', '8087AL(41)']
  '7306AL-7307AL(40-41)'  ->  2 row(s): ['7306AL(40)', '7307AL(41)']

6 original rows expanded into 14 individual row

## 2H\. Clean bag number in QC file

In [34]:
# ---- Create bag_no_corrected in qc_df (must run before Block 5) ----
MONTH_MAP = {
    'jan': 1, 'feb': 2, 'mar': 3, 'apr': 4, 'may': 5, 'jun': 6,
    'jul': 7, 'aug': 8, 'sep': 9, 'oct': 10, 'nov': 11, 'dec': 12
}

def fix_bag_no(value):
    if pd.isna(value) or str(value).strip() == '':
        return None
    value = str(value).strip()
    def replace_month(match):
        return str(MONTH_MAP[match.group(0).lower()])
    fixed = re.sub(r'(?i)(jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)', replace_month, value)
    parts = fixed.split('-')
    if len(parts) == 2:
        try:
            a, b = int(parts[0].strip()), int(parts[1].strip())
            return f'{min(a,b)}-{max(a,b)}'
        except:
            pass
    return fixed

qc_df["bag_no_corrected"] = qc_df["bag_no"].apply(fix_bag_no)
print("bag_no_corrected created.")

changed = qc_df[qc_df["bag_no"].astype(str).str.strip() != qc_df["bag_no_corrected"].astype(str).str.strip()]
print(len(changed))

bag_no_corrected created.
11974


CLEAN BAG NUMBER IN REMARKS COLUMN

In [35]:
# List all bags in remarks column
BAG_WORD_PATTERN = re.compile(r'\bbag\b', re.IGNORECASE)

has_bag_in_remarks = qc_df["remarks"].apply(
    lambda x: bool(BAG_WORD_PATTERN.search(str(x))) if pd.notna(x) else False
)

subset = qc_df.loc[has_bag_in_remarks, ["id", "product_code", "lot_number", "remarks"]]
print(f"Records with 'bag' mentioned in remarks: {len(subset)}\n")
print(subset.to_string(index=False, max_colwidth=100))

Records with 'bag' mentioned in remarks: 210

   id product_code      lot_number                                                                                              remarks
13716     BA12556E  6124AM(6127AM)                                         color pass, but observed dirt & blue spots; megabag#4 bag#70
11034     BA12556E          7957AN                                                                                         mega bag # 4
11033     BA12556E          7957AN                                                                                         mega bag # 3
11032     BA12556E          7957AN                                                                                         mega bag # 2
11031     BA12556E          7957AN                                                                                         mega bag # 1
10788      WA3827E          7158AN                                                                change lot/re bag of 5289AN; & 5970AN
10

In [36]:
def normalize_bag_spacing(remarks_value):
    if pd.isna(remarks_value):
        return remarks_value
    text = str(remarks_value)
    text = re.sub(r'(?i)\bmega\s+bag\b', 'megabag', text)  # "mega bag", "mega  bag" -> "megabag"
    text = re.sub(r'(?i)\bbag\s*#\s*', 'bag#', text)        # "bag # 1", "bag  #1" -> "bag#1"
    text = re.sub(r'(?i)#\s+', '#', text)                    # any remaining "# 1" -> "#1"
    return text

qc_df["remarks_normalized"] = qc_df["remarks"].apply(normalize_bag_spacing)

changed_spacing = qc_df[qc_df["remarks"].astype(str) != qc_df["remarks_normalized"].astype(str)]
print(f"Remarks with spacing normalized: {len(changed_spacing)}\n")
print(changed_spacing[["id", "remarks", "remarks_normalized"]].to_string(index=False, max_colwidth=50))

Remarks with spacing normalized: 98

   id                                            remarks                                 remarks_normalized
13511 No sample submitted for lot# 3752AO-3753AO (los... No sample submitted for lot#3752AO-3753AO (loss...
11034                                       mega bag # 4                                         megabag #4
11033                                       mega bag # 3                                         megabag #3
11032                                       mega bag # 2                                         megabag #2
11031                                       mega bag # 1                                         megabag #1
 6745        color pass; observed dark spots; megabag# 4         color pass; observed dark spots; megabag#4
 6210        color pass; observed blue spots; BAG#10-11;        color pass; observed blue spots; bag#10-11;
 6209        color pass; observed blue spots; BAG#50-51;        color pass; observed blue spots; ba

In [37]:
BAG_REMARKS_PATTERN = re.compile(
    r'\bbag\b\s*#?\s*(\d{1,3}(?:\s*-\s*\d{1,3})?)',
    re.IGNORECASE
)

def extract_bag_from_remarks(remarks_value):
    if pd.isna(remarks_value):
        return None
    match = BAG_REMARKS_PATTERN.search(str(remarks_value))
    if match:
        return match.group(1).replace(' ', '')
    return None

needs_fallback = qc_df["bag_no_corrected"].isna() | (qc_df["bag_no_corrected"].astype(str).str.strip() == '')
qc_df.loc[needs_fallback, "bag_no_corrected"] = qc_df.loc[needs_fallback, "remarks_normalized"].apply(extract_bag_from_remarks)

recovered = qc_df.loc[needs_fallback & qc_df["bag_no_corrected"].notna()]
print(f"Bag numbers recovered from remarks: {len(recovered)}\n")
print(recovered[["id", "product_code", "lot_number", "remarks_normalized", "bag_no_corrected"]].to_string(index=False, max_colwidth=80))

Bag numbers recovered from remarks: 199

   id product_code      lot_number                                                               remarks_normalized bag_no_corrected
13716     BA12556E  6124AM(6127AM)                     color pass, but observed dirt & blue spots; megabag#4 bag#70               70
 6210     BA17042E          7196AM                                      color pass; observed blue spots; bag#10-11;            10-11
 6209     BA17042E  7196AM(7198AM)                                      color pass; observed blue spots; bag#50-51;            50-51
 6207     BA17042E  7196AM(7197AM)                                       color pass; observed blue spots; bag#30-31            30-31
 5803     BA12556E  6344AM(6345AM)            color pass; observed dark spots; correction of 5793AM(4); kraft bag#3                3
 5769     WA12282E  6240AM(6249AM)                 color pass; observed dirt embedded in pellet; megabag#10 bag#190              190
 5763     WA12282E  6240AM(6

In [38]:
def fix_mega_typos(remarks_value):
    if pd.isna(remarks_value):
        return remarks_value
    text = str(remarks_value)
    text = re.sub(r'(?i)\bmaga\s+bag\b', 'megabag', text)  # "maga bag" -> "megabag"
    text = re.sub(r'(?i)\bmeg\s+bag\b', 'megabag', text)   # "meg bag" -> "megabag"
    return text

qc_df["remarks_normalized"] = qc_df["remarks_normalized"].apply(fix_mega_typos)

# Re-run the extraction now that the typos are fixed
qc_df.loc[needs_fallback, "bag_no_corrected"] = qc_df.loc[needs_fallback, "remarks_normalized"].apply(extract_bag_from_remarks)

recovered = qc_df.loc[needs_fallback & qc_df["bag_no_corrected"].notna()]
print(f"Bag numbers recovered from remarks (after typo fix): {len(recovered)}\n")
print(recovered[["id", "product_code", "lot_number", "remarks_normalized", "bag_no_corrected"]].to_string(index=False, max_colwidth=50))

Bag numbers recovered from remarks (after typo fix): 199

   id product_code      lot_number                                 remarks_normalized bag_no_corrected
13716     BA12556E  6124AM(6127AM) color pass, but observed dirt & blue spots; meg...               70
 6210     BA17042E          7196AM        color pass; observed blue spots; bag#10-11;            10-11
 6209     BA17042E  7196AM(7198AM)        color pass; observed blue spots; bag#50-51;            50-51
 6207     BA17042E  7196AM(7197AM)         color pass; observed blue spots; bag#30-31            30-31
 5803     BA12556E  6344AM(6345AM) color pass; observed dark spots; correction of ...                3
 5769     WA12282E  6240AM(6249AM) color pass; observed dirt embedded in pellet; m...              190
 5763     WA12282E  6240AM(6248AM) color pass; observed dirt embedded in pellet; m...              170
 5762     WA12282E  6240AM(6247AM) color pass; observed dirt embedded in pellet; m...              150
 5761     WA122

In [39]:
VALID_BAG_FORMAT = re.compile(r'^\d{1,3}(-\d{1,3})?$')

verify_df = qc_df.loc[needs_fallback & qc_df["bag_no_corrected"].notna()].copy()
verify_df["format_valid"] = verify_df["bag_no_corrected"].astype(str).str.match(VALID_BAG_FORMAT)
verify_df["status"] = verify_df["format_valid"].map({True: "OK", False: "FAILED"})

print(f"Total bag numbers recovered from remarks: {len(verify_df)}")
print(f"Passed format validation: {verify_df['format_valid'].sum()}")
print(f"Failed format validation: {(~verify_df['format_valid']).sum()}\n")

print(verify_df[["id", "product_code", "lot_number", "remarks_normalized", "bag_no_corrected", "status"]].to_string(index=False, max_colwidth=50))

Total bag numbers recovered from remarks: 199
Passed format validation: 199
Failed format validation: 0

   id product_code      lot_number                                 remarks_normalized bag_no_corrected status
13716     BA12556E  6124AM(6127AM) color pass, but observed dirt & blue spots; meg...               70     OK
 6210     BA17042E          7196AM        color pass; observed blue spots; bag#10-11;            10-11     OK
 6209     BA17042E  7196AM(7198AM)        color pass; observed blue spots; bag#50-51;            50-51     OK
 6207     BA17042E  7196AM(7197AM)         color pass; observed blue spots; bag#30-31            30-31     OK
 5803     BA12556E  6344AM(6345AM) color pass; observed dark spots; correction of ...                3     OK
 5769     WA12282E  6240AM(6249AM) color pass; observed dirt embedded in pellet; m...              190     OK
 5763     WA12282E  6240AM(6248AM) color pass; observed dirt embedded in pellet; m...              170     OK
 5762     WA122

# \(B\) SPECTRO FILES VALIDATION

## 3\. Load SPECTRO file \(raw, no changes yet\)

In [40]:
# ---- Step 3: Load Spectro files (raw, no changes yet) ----
spectro_folder = "from_spectro"
spectro_files = glob.glob(os.path.join(spectro_folder, "*.xlsx"))

spectro_data = {}       # product_code -> dataframe
spectro_filepaths = {}  # product_code -> original full filepath (for traceability downstream)

print("Spectro files found:")
for f in spectro_files:
    filename = os.path.basename(f)
    name_no_ext = os.path.splitext(filename)[0]  # strip extension first
    product_code = name_no_ext.split(" ")[0]     # text before first space, extension-safe
    df = pd.read_excel(f)
    spectro_data[product_code] = df
    spectro_filepaths[product_code] = f
    print(f" - {product_code}: {len(df)} rows, file = '{filename}'")

print(f"\nTotal Spectro files loaded: {len(spectro_data)}")

Spectro files found:
 - BA0830E: 123 rows, file = 'BA0830E from_spectro.xlsx'
 - BA11939E: 5 rows, file = 'BA11939E from_spectro.xlsx'
 - BA11959E: 5 rows, file = 'BA11959E from_spectro.xlsx'
 - BA12013E: 18 rows, file = 'BA12013E from_spectro.xlsx'
 - BA12068E: 24 rows, file = 'BA12068E from_spectro.xlsx'
 - BA12615E: 63 rows, file = 'BA12615E from_spectro.xlsx'
 - BA12618E: 39 rows, file = 'BA12618E from_spectro.xlsx'
 - BA12855E: 4 rows, file = 'BA12855E from_spectro.xlsx'
 - BA12861E: 38 rows, file = 'BA12861E from_spectro.xlsx'
 - BA13023E: 107 rows, file = 'BA13023E from_spectro.xlsx'
 - BA13652E: 6 rows, file = 'BA13652E from_spectro.xlsx'
 - BA14168E: 6 rows, file = 'BA14168E from_spectro.xlsx'
 - BA14304E: 65 rows, file = 'BA14304E from_spectro.xlsx'
 - BA14432E: 331 rows, file = 'BA14432E from_spectro.xlsx'
 - BA14456E: 100 rows, file = 'BA14456E from_spectro.xlsx'
 - BA15197E: 149 rows, file = 'BA15197E from_spectro.xlsx'
 - BA15494E: 58 rows, file = 'BA15494E from_spectro.x

In [41]:
# ---- Step 2H (new): Drop fully-blank separator rows from raw Spectro data ----
# These are structural export artifacts (blank row before a new STD/reference block),
# not real data — confirmed against BA12615E and BA12861E raw files. A row with
# every column NaN can never be a real reading, so drop it before any normalization
# touches the Name column (this also prevents the NaN->"NAN" string bug downstream).

for code, df in spectro_data.items():
    before = len(df)
    df = df.dropna(how="all").reset_index(drop=True)
    dropped = before - len(df)
    spectro_data[code] = df
    if dropped:
        print(f"{code}: dropped {dropped} fully-blank row(s)")

BA12615E: dropped 1 fully-blank row(s)
BA13023E: dropped 1 fully-blank row(s)
BA15197E: dropped 1 fully-blank row(s)
BA15494E: dropped 1 fully-blank row(s)
BA17933E: dropped 1 fully-blank row(s)


## 3A\. Normalize Name column \(Column B\) before validation

In [42]:
# ---- Step 3A: Normalize Column B (Name) per Spectro file ----
for code, df in spectro_data.items():
    name_col = "Name"  # Column B

    # Preserve the untouched raw value before any changes — same pattern as QC's raw column
    df["Name_raw"] = df[name_col]

    df[name_col] = (
        df[name_col]
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)  # collapse multiple spaces to one (not remove entirely — spacing is meaningful here)
        .str.strip()
        .str.upper()
    )

print("Column B normalized (uppercase, trimmed, single spaces) for all Spectro files.")
print("Original raw values preserved in 'Name_raw' column.")


Column B normalized (uppercase, trimmed, single spaces) for all Spectro files.
Original raw values preserved in 'Name_raw' column.


In [43]:
# ---- Step 3A (addendum): Print normalized Column B values per Spectro file ----
for code, df in spectro_data.items():
    name_col = "Name"
    print(f"\n--- {code} ---")
    print(f"Total rows: {len(df)}")
    print(df[name_col].tolist())


--- BA0830E ---
Total rows: 123
['BA0830E 2260P 0.4% STD', '4484AL 0.4% LIGHT', '2193P 0.4% DARK', '4484AL', '4528AL', '4529AL', '4530AL', '4531AL', '4532AL', '4533AL', '4534AL', '4535AL', '4536AL', '4537AL', '4538AL', '4539AL', '4540AL', '4541AL', '8264AL', '8265AL', '8266AL', '8267AL', '8268AL', '8269AL', '8270AL', '8271AL', '8272AL', '8273AL', '8274AL', '8275AL', '8276AL', '8780AL', '8781AL', '8782AL', '8783AL', '8784AL', '8785AL', '8786AL', '8787AL', '8788AL', '8789AL', '8790AL', '8791AL', '8792AL', '8793AL', '8794AL', '8795AL', '8796AL', '8797AL', '8798AL', '8799AL', '8800AL', '8801AL', '8802AL', '8803AL', '8804AL', '8805AL', '8806AL 28 OS', '6359AM', '6360AM', '6361AM', '6362AM', '5342AN', '5343AN', '5344AN', '5345AN', '5346AN', '5347AN', '5347AN 7 OS', '5801AN', '5802AN', '5803AN', '5804AN', '5805AN', '5806AN', '5807AN', '5808AN', '5809AN', '5810AN', '5811AN', '5812AN', '5813AN', '5814AN', '5815AN', '5816AN', '5817AN', '5818AN', '5819AN', '5820AN', '5821AN', '5822AN', '5823AN',

## 3B\. Clean reference rows

### \- 3B Validation Check 1: Identify reference rows

In [44]:
# ---- Identify reference rows (STD, LIGHT, DARK, CMA, %) ----
def is_reference_row(value):
    if pd.isna(value):
        return True
    value = str(value).upper()
    return any(marker in value for marker in ["STD", "LIGHT", "DARK", "CMA", "%"])

for code, df in spectro_data.items():
    name_col = "Name"
    df["is_reference_row"] = df[name_col].apply(is_reference_row)

    ref_count = df["is_reference_row"].sum()
    print(f"{code}: {ref_count} reference row(s) identified out of {len(df)} total")

BA0830E: 3 reference row(s) identified out of 123 total
BA11939E: 3 reference row(s) identified out of 5 total
BA11959E: 3 reference row(s) identified out of 5 total
BA12013E: 3 reference row(s) identified out of 18 total
BA12068E: 3 reference row(s) identified out of 24 total
BA12615E: 6 reference row(s) identified out of 62 total
BA12618E: 3 reference row(s) identified out of 39 total
BA12855E: 3 reference row(s) identified out of 4 total
BA12861E: 3 reference row(s) identified out of 38 total
BA13023E: 6 reference row(s) identified out of 106 total
BA13652E: 3 reference row(s) identified out of 6 total
BA14168E: 3 reference row(s) identified out of 6 total
BA14304E: 3 reference row(s) identified out of 65 total
BA14432E: 3 reference row(s) identified out of 331 total
BA14456E: 3 reference row(s) identified out of 100 total
BA15197E: 4 reference row(s) identified out of 148 total
BA15494E: 6 reference row(s) identified out of 57 total
BA15965E: 3 reference row(s) identified out of 14

In [45]:
for code, df in spectro_data.items():
    name_col = "Name"
    ref_rows = df[df["is_reference_row"] == True]

    print(f"\n--- {code} ---")
    print(f"Reference row(s) found: {len(ref_rows)}")
    print(ref_rows[name_col].tolist())


--- BA0830E ---
Reference row(s) found: 3
['BA0830E 2260P 0.4% STD', '4484AL 0.4% LIGHT', '2193P 0.4% DARK']

--- BA11939E ---
Reference row(s) found: 3
['BA11939E CMA-09995 4.0% STD', '1637AI 4.0% LIGHT', '8831AD 4.0% DARK']

--- BA11959E ---
Reference row(s) found: 3
['BA11959E CMA-10053 4.0% STD', '8561AD 4.0% LIGHT', '1450AI 4.0% DARK']

--- BA12013E ---
Reference row(s) found: 3
['BA12013E CMA-10059 4.0% STD', '3640AD 4.0% LIGHT', '0210AH 4.0% DARK']

--- BA12068E ---
Reference row(s) found: 3
['BA12068E CMA-08892 3.0% STD', '0338AC 3.0% LIGHT', '2097AC 3.0% DARK']

--- BA12615E ---
Reference row(s) found: 6
['BA12615E CMA-10500 3.0% STD', '0703AE 3.0% LIGHT', '7247AE 3.0% DARK', 'BA12615E CMA-10500 3.0% STD UPDATED', '0703AE 3.0% LIGHT UPDATED', '7247AE 3.0% DARK UPDATED']

--- BA12618E ---
Reference row(s) found: 3
['BA12618E CMA-10504 3.0% STD', '9513AD 3.0% LIGHT', '2419AJ 3.0% DARK']

--- BA12855E ---
Reference row(s) found: 3
['BA12855E 0661AG 3.0% STD', 'CMA-10725 3.0 % LI

### \- 3B Re\-arrangement of lot numbers & product codes

In [46]:
# Split reference rows into product_code and lot_number ----

REFERENCE_MARKER_PATTERN = re.compile(r"\b(STD|LIGHT|DARK)\b(\s+UPDATED\b)?", re.IGNORECASE)
MARKER_SHORT = {"STD": "STD", "LIGHT": "LT", "DARK": "DR"}

def clean_reference_lot_number(value, product_code):
    value = str(value).strip().upper()
    code_upper = str(product_code).strip().upper()

    if value.startswith(code_upper):
        value = value[len(code_upper):].strip()

    match = REFERENCE_MARKER_PATTERN.search(value)
    if match:
        marker = match.group(1).upper()
        has_updated = bool(match.group(2))
        remainder = (value[:match.start()] + value[match.end():]).strip()
        remainder = re.sub(r"\s+", " ", remainder)

        short = MARKER_SHORT[marker]
        prefix = f"{short} UPDATED" if has_updated else short
        value = f"{prefix} {remainder}".strip()

    return value


for code, df in spectro_data.items():
    ref_mask = df["is_reference_row"] == True
    name_col = "Name"

    # Create product_code column beside Name, blank for non-reference rows for now
    if "product_code" not in df.columns:
        df.insert(df.columns.get_loc(name_col) + 1, "product_code", None)

    for idx in df[ref_mask].index:
        df.loc[idx, "product_code"] = code  # the file's own product code
        df.loc[idx, name_col] = clean_reference_lot_number(df.loc[idx, name_col], code)

    # Rename Name column to "lot number"
    df.rename(columns={name_col: "lot number"}, inplace=True)

    print(f"\n--- {code} ---")
    print(df.loc[ref_mask, ["lot number", "product_code"]].to_string(index=False))


--- BA0830E ---
    lot number product_code
STD 2260P 0.4%      BA0830E
LT 4484AL 0.4%      BA0830E
 DR 2193P 0.4%      BA0830E

--- BA11939E ---
        lot number product_code
STD CMA-09995 4.0%     BA11939E
    LT 1637AI 4.0%     BA11939E
    DR 8831AD 4.0%     BA11939E

--- BA11959E ---
        lot number product_code
STD CMA-10053 4.0%     BA11959E
    LT 8561AD 4.0%     BA11959E
    DR 1450AI 4.0%     BA11959E

--- BA12013E ---
        lot number product_code
STD CMA-10059 4.0%     BA12013E
    LT 3640AD 4.0%     BA12013E
    DR 0210AH 4.0%     BA12013E

--- BA12068E ---
        lot number product_code
STD CMA-08892 3.0%     BA12068E
    LT 0338AC 3.0%     BA12068E
    DR 2097AC 3.0%     BA12068E

--- BA12615E ---
                lot number product_code
        STD CMA-10500 3.0%     BA12615E
            LT 0703AE 3.0%     BA12615E
            DR 7247AE 3.0%     BA12615E
STD UPDATED CMA-10500 3.0%     BA12615E
    LT UPDATED 0703AE 3.0%     BA12615E
    DR UPDATED 7247AE 3.0%   

## 3C\. Validate Column B \(Name\)

### \- 3C Differentiate Lot Numbers to Reference Lot Numbers

In [47]:
# ---- Print total row counts and reference row counts across all Spectro files ----

total_rows_all_files = 0
total_reference_rows_all_files = 0

for code, df in spectro_data.items():
    total_rows_all_files += len(df)
    total_reference_rows_all_files += df["is_reference_row"].sum()

print(f"Total rows across all Spectro files: {total_rows_all_files}")
print(f"Total reference rows across all Spectro files: {total_reference_rows_all_files}")
print(f"Total real (non-reference) rows: {total_rows_all_files - total_reference_rows_all_files}")

Total rows across all Spectro files: 2278
Total reference rows across all Spectro files: 110
Total real (non-reference) rows: 2168


### \- 3C Identify valid and invalid lot number formats

In [48]:
# put all lot data in a list
valid_spectro_names = []
invalid_spectro_names = []

In [49]:
# ---- Step 3C Block 1: Identify valid vs invalid lot number formats (real rows only) ----
import re

# Confirmed formats:
#   0000XX          — lot number only
#   0000XX 9        — lot + single-digit bag
#   0000XX 99       — lot + double-digit bag
#   0000XX 999      — lot + triple-digit bag
#   0000XX 99-100   — lot + bag range
#   0000XX 99 OS    — lot + bag + OS
#   0000XX 999 OS   — lot + triple-digit bag + OS
#   0000XX OS       — lot + OS, no bag
SPECTRO_NAME_PATTERN = re.compile(
    r"^\d{4}[A-Za-z]{1,2}(\s\d{1,3}(-\d{1,3})?)?(\sOS)?$"
)

# valid_spectro_names = []
# invalid_spectro_names = []

for code, df in spectro_data.items():
    real_rows = df[df["is_reference_row"] == False]

    for idx, row in real_rows.iterrows():
        value = row["lot number"]
        value_str = str(value).strip() if not pd.isna(value) else ""

        if not pd.isna(value) and SPECTRO_NAME_PATTERN.match(value_str):
            valid_spectro_names.append({"product_code": code, "row_index": idx, "value": value_str})
        else:
            invalid_spectro_names.append({"product_code": code, "row_index": idx, "value": value_str})

print("Valid lot number values:")
for entry in valid_spectro_names:
    print(f"  [{entry['product_code']}] Row {entry['row_index']}: '{entry['value']}'")

Valid lot number values:
  [BA0830E] Row 3: '4484AL'
  [BA0830E] Row 4: '4528AL'
  [BA0830E] Row 5: '4529AL'
  [BA0830E] Row 6: '4530AL'
  [BA0830E] Row 7: '4531AL'
  [BA0830E] Row 8: '4532AL'
  [BA0830E] Row 9: '4533AL'
  [BA0830E] Row 10: '4534AL'
  [BA0830E] Row 11: '4535AL'
  [BA0830E] Row 12: '4536AL'
  [BA0830E] Row 13: '4537AL'
  [BA0830E] Row 14: '4538AL'
  [BA0830E] Row 15: '4539AL'
  [BA0830E] Row 16: '4540AL'
  [BA0830E] Row 17: '4541AL'
  [BA0830E] Row 18: '8264AL'
  [BA0830E] Row 19: '8265AL'
  [BA0830E] Row 20: '8266AL'
  [BA0830E] Row 21: '8267AL'
  [BA0830E] Row 22: '8268AL'
  [BA0830E] Row 23: '8269AL'
  [BA0830E] Row 24: '8270AL'
  [BA0830E] Row 25: '8271AL'
  [BA0830E] Row 26: '8272AL'
  [BA0830E] Row 27: '8273AL'
  [BA0830E] Row 28: '8274AL'
  [BA0830E] Row 29: '8275AL'
  [BA0830E] Row 30: '8276AL'
  [BA0830E] Row 31: '8780AL'
  [BA0830E] Row 32: '8781AL'
  [BA0830E] Row 33: '8782AL'
  [BA0830E] Row 34: '8783AL'
  [BA0830E] Row 35: '8784AL'
  [BA0830E] Row 36: '8785

In [50]:
print(f"Valid lot number format count: {len(valid_spectro_names)}")

Valid lot number format count: 2165


### \- 3C Get invalid Lot Number format

In [51]:
# ---- Step 3C Block 2: Print invalid lot number formats ----
if invalid_spectro_names:
    print(f"FLAGGED: {len(invalid_spectro_names)} row(s) with unexpected Column B format.")
    for issue in invalid_spectro_names:
        print(f"  [{issue['product_code']}] Row {issue['row_index']}: '{issue['value']}'")
else:
    print("All Spectro Column B values passed validation.")

FLAGGED: 3 row(s) with unexpected Column B format.
  [BA17935E] Row 5: '4560AN 2ND XS'
  [BA17935E] Row 6: '4561AN 2ND XS'
  [WA15190E] Row 115: '9317AN 249-250; OS'


TEST BLOCK: TRY APPLYING CORRECTIONS\.

In [52]:
# ---- Apply confirmed corrections for Spectro Column B anomalies ----
SPECTRO_NAME_CORRECTIONS = {
    "WA15190E": {
        115: "9317AN 249-250 OS",
    }
}

for code, df in spectro_data.items():
    if "lot number_corrected" not in df.columns:
        df["lot number_corrected"] = df["lot number"]

    if code in SPECTRO_NAME_CORRECTIONS:
        for row_idx, corrected_value in SPECTRO_NAME_CORRECTIONS[code].items():
            if row_idx in df.index:
                df.loc[row_idx, "lot number_corrected"] = corrected_value

corrected = spectro_data["WA15190E"].loc[115, ["lot number", "lot number_corrected"]]
print(corrected)

lot number              9317AN 249-250; OS
lot number_corrected     9317AN 249-250 OS
Name: 115, dtype: object


In [53]:
# ---- Step 3C Block 4: Re-validate after corrections, using lot number_corrected ----

valid_spectro_names_corrected = []
invalid_spectro_names_corrected = []

for code, df in spectro_data.items():
    real_rows = df[df["is_reference_row"] == False]

    for idx, row in real_rows.iterrows():
        value = row["lot number_corrected"]
        value_str = str(value).strip() if not pd.isna(value) else ""

        if not pd.isna(value) and SPECTRO_NAME_PATTERN.match(value_str):
            valid_spectro_names_corrected.append({"product_code": code, "row_index": idx, "value": value_str})
        else:
            invalid_spectro_names_corrected.append({"product_code": code, "row_index": idx, "value": value_str})

print(f"Valid lot number format count (after corrections): {len(valid_spectro_names_corrected)}")

if invalid_spectro_names_corrected:
    print(f"FLAGGED: {len(invalid_spectro_names_corrected)} row(s) still with unexpected format:")
    for issue in invalid_spectro_names_corrected:
        print(f"  [{issue['product_code']}] Row {issue['row_index']}: '{issue['value']}'")
else:
    print("All Spectro Column B values passed validation after corrections.")

Valid lot number format count (after corrections): 2166
FLAGGED: 2 row(s) still with unexpected format:
  [BA17935E] Row 5: '4560AN 2ND XS'
  [BA17935E] Row 6: '4561AN 2ND XS'


## 3D\. Parse Spectro File

### \- 3D Step 1: Cleanse Spectro Columns

In [54]:
# ---- Step 3D Block 1: Call and display current columns in each Spectro dataframe ----

for code, df in spectro_data.items():
    print(f"\n--- {code} ---")
    print(df.columns.tolist())


--- BA0830E ---
['Color Simulation', 'Date Time', 'lot number', 'product_code', 'ΔE*00', 'L*', 'C*', 'h°', 'a*', 'b*', 'ΔL*', 'ΔC*', 'ΔH*', 'Δa*', 'Δb*', 'Color Offset', 'Judgement', 'Name_raw', 'is_reference_row', 'lot number_corrected']

--- BA11939E ---
['Color Simulation', 'Date Time', 'lot number', 'product_code', 'ΔE*00', 'L*', 'C*', 'h°', 'a*', 'b*', 'ΔL*', 'ΔC*', 'ΔH*', 'Δa*', 'Δb*', 'Color Offset', 'Judgement', 'Name_raw', 'is_reference_row', 'lot number_corrected']

--- BA11959E ---
['Color Simulation', 'Date Time', 'lot number', 'product_code', 'ΔE*00', 'L*', 'C*', 'h°', 'a*', 'b*', 'ΔL*', 'ΔC*', 'ΔH*', 'Δa*', 'Δb*', 'Color Offset', 'Judgement', 'Name_raw', 'is_reference_row', 'lot number_corrected']

--- BA12013E ---
['Color Simulation', 'Date Time', 'lot number', 'product_code', 'ΔE*00', 'L*', 'C*', 'h°', 'a*', 'b*', 'ΔL*', 'ΔC*', 'ΔH*', 'Δa*', 'Δb*', 'Color Offset', 'Judgement', 'Opacity', 'Name_raw', 'is_reference_row', 'lot number_corrected']

--- BA12068E ---
['Color 

In [55]:
# ---- Rename columns back for clarity ----
for code, df in spectro_data.items():
    df.rename(columns={
        "lot number": "Name",
        "product_code": "Product Code",
        "lot number_corrected": "Name_corrected"
    }, inplace=True)

In [56]:
# ---- Apply correction directly into Name, drop Name_corrected ----
for code, df in spectro_data.items():
    if "Name_corrected" in df.columns:
        df["Name"] = df["Name_corrected"]
        df.drop(columns=["Name_corrected"], inplace=True)

for code, df in spectro_data.items():
    print(f"\n--- {code} ---")
    print(df.columns.tolist())


--- BA0830E ---
['Color Simulation', 'Date Time', 'Name', 'Product Code', 'ΔE*00', 'L*', 'C*', 'h°', 'a*', 'b*', 'ΔL*', 'ΔC*', 'ΔH*', 'Δa*', 'Δb*', 'Color Offset', 'Judgement', 'Name_raw', 'is_reference_row']

--- BA11939E ---
['Color Simulation', 'Date Time', 'Name', 'Product Code', 'ΔE*00', 'L*', 'C*', 'h°', 'a*', 'b*', 'ΔL*', 'ΔC*', 'ΔH*', 'Δa*', 'Δb*', 'Color Offset', 'Judgement', 'Name_raw', 'is_reference_row']

--- BA11959E ---
['Color Simulation', 'Date Time', 'Name', 'Product Code', 'ΔE*00', 'L*', 'C*', 'h°', 'a*', 'b*', 'ΔL*', 'ΔC*', 'ΔH*', 'Δa*', 'Δb*', 'Color Offset', 'Judgement', 'Name_raw', 'is_reference_row']

--- BA12013E ---
['Color Simulation', 'Date Time', 'Name', 'Product Code', 'ΔE*00', 'L*', 'C*', 'h°', 'a*', 'b*', 'ΔL*', 'ΔC*', 'ΔH*', 'Δa*', 'Δb*', 'Color Offset', 'Judgement', 'Opacity', 'Name_raw', 'is_reference_row']

--- BA12068E ---
['Color Simulation', 'Date Time', 'Name', 'Product Code', 'ΔE*00', 'L*', 'C*', 'h°', 'a*', 'b*', 'ΔL*', 'ΔC*', 'ΔH*', 'Δa*', 'Δb

In [57]:
# ---- Print full data per product code after renaming ----
for code, df in spectro_data.items():
    print(f"\n--- {code} ({len(df)} rows) ---")
    print(df.to_string(index=False))


--- BA0830E (123 rows) ---
Color Simulation           Date Time           Name Product Code ΔE*00    L*    C*     h°    a*     b*   ΔL*   ΔC*   ΔH*   Δa*   Δb*          Color Offset Judgement               Name_raw  is_reference_row
       #FF2F5790 2024-11-20 11:42:51 STD 2260P 0.4%      BA0830E    -- 36.19 36.36 271.19  0.76 -36.35    --    --    --    --    --                    --        -- BA0830E 2260P 0.4% STD              True
       #FF2F568E 2024-11-20 11:42:54 LT 4484AL 0.4%      BA0830E  0.24 35.93 35.92 271.05  0.66 -35.92 -0.26 -0.44 -0.09  -0.1  0.43                   NaN      Pass      4484AL 0.4% LIGHT              True
       #FF325489 2024-11-20 11:43:00  DR 2193P 0.4%      BA0830E  1.53 35.14 34.03 272.09  1.24 -34.00 -1.05 -2.33  0.55  0.48  2.34          Dark+, Blue-      Fail        2193P 0.4% DARK              True
       #FF2D5692 2024-11-20 11:43:31         4484AL         None  0.44 36.01 38.13 271.96  1.31 -38.10 -0.18  1.77   0.5  0.55 -1.75           Red+,

### \- 3D Step 2: Splitting values

In [58]:
for code, df in spectro_data.items():
    real_count = (df["is_reference_row"] == False).sum()
    print(f"{code}: {real_count} row(s) (excluding reference rows)")

BA0830E: 120 row(s) (excluding reference rows)
BA11939E: 2 row(s) (excluding reference rows)
BA11959E: 2 row(s) (excluding reference rows)
BA12013E: 15 row(s) (excluding reference rows)
BA12068E: 21 row(s) (excluding reference rows)
BA12615E: 56 row(s) (excluding reference rows)
BA12618E: 36 row(s) (excluding reference rows)
BA12855E: 1 row(s) (excluding reference rows)
BA12861E: 35 row(s) (excluding reference rows)
BA13023E: 100 row(s) (excluding reference rows)
BA13652E: 3 row(s) (excluding reference rows)
BA14168E: 3 row(s) (excluding reference rows)
BA14304E: 62 row(s) (excluding reference rows)
BA14432E: 328 row(s) (excluding reference rows)
BA14456E: 97 row(s) (excluding reference rows)
BA15197E: 144 row(s) (excluding reference rows)
BA15494E: 51 row(s) (excluding reference rows)
BA15965E: 139 row(s) (excluding reference rows)
BA16168E: 3 row(s) (excluding reference rows)
BA16776E: 65 row(s) (excluding reference rows)
BA17452E: 5 row(s) (excluding reference rows)
BA17468E: 11 row

In [59]:
# ---- Block 1: Fill Product Code for non-reference (real) rows ----
for code, df in spectro_data.items():
    df["Product Code"] = code

for code, df in spectro_data.items():
    print(f"\n--- {code} ---")
    print(df[["Name", "Product Code", "is_reference_row"]].to_string(index=False))


--- BA0830E ---
          Name Product Code  is_reference_row
STD 2260P 0.4%      BA0830E              True
LT 4484AL 0.4%      BA0830E              True
 DR 2193P 0.4%      BA0830E              True
        4484AL      BA0830E             False
        4528AL      BA0830E             False
        4529AL      BA0830E             False
        4530AL      BA0830E             False
        4531AL      BA0830E             False
        4532AL      BA0830E             False
        4533AL      BA0830E             False
        4534AL      BA0830E             False
        4535AL      BA0830E             False
        4536AL      BA0830E             False
        4537AL      BA0830E             False
        4538AL      BA0830E             False
        4539AL      BA0830E             False
        4540AL      BA0830E             False
        4541AL      BA0830E             False
        8264AL      BA0830E             False
        8265AL      BA0830E             False
        8266AL   

In [60]:
# ---- Block 2: Split Name into lot number + Bag No., reorder columns ----

def split_name_bag(value, is_ref):
    """Splits a real-row Name value into (lot_number, bag_no_str). Reference rows untouched."""
    if is_ref:
        return value, None

    value = str(value).strip()
    is_os = value.upper().endswith(" OS")
    if is_os:
        value = value[:-3].strip()

    parts = value.split(" ", 1)
    lot_part = parts[0].strip()
    bag_part = parts[1].strip() if len(parts) > 1 else None

    # Reattach OS marker temporarily so Block 4 can detect it before removal
    if is_os:
        bag_part = f"{bag_part} OS" if bag_part else "OS"

    return lot_part, bag_part


for code, df in spectro_data.items():
    split_results = df.apply(lambda row: split_name_bag(row["Name"], row["is_reference_row"]), axis=1)
    df["Name"] = [r[0] for r in split_results]
    df["Bag No."] = [r[1] for r in split_results]

    # Reorder columns: Color Simulation, Product Code, Name, Bag No., then the rest
    cols = list(df.columns)
    front = ["Color Simulation", "Product Code", "Name", "Bag No."]
    remaining = [c for c in cols if c not in front]
    df_cols_ordered = front + remaining
    spectro_data[code] = df[df_cols_ordered]

for code, df in spectro_data.items():
    print(f"\n--- {code} ---")
    print(df[["Color Simulation", "Product Code", "Name", "Bag No."]].to_string(index=False))


--- BA0830E ---
Color Simulation Product Code           Name Bag No.
       #FF2F5790      BA0830E STD 2260P 0.4%    None
       #FF2F568E      BA0830E LT 4484AL 0.4%    None
       #FF325489      BA0830E  DR 2193P 0.4%    None
       #FF2D5692      BA0830E         4484AL    None
       #FF2F5691      BA0830E         4528AL    None
       #FF325A94      BA0830E         4529AL    None
       #FF2E5792      BA0830E         4530AL    None
       #FF2F5690      BA0830E         4531AL    None
       #FF2E5591      BA0830E         4532AL    None
       #FF2C5693      BA0830E         4533AL    None
       #FF2E5692      BA0830E         4534AL    None
       #FF2E5793      BA0830E         4535AL    None
       #FF2D5692      BA0830E         4536AL    None
       #FF2D5591      BA0830E         4537AL    None
       #FF2D5591      BA0830E         4538AL    None
       #FF2E548E      BA0830E         4539AL    None
       #FF2C548F      BA0830E         4540AL    None
       #FF2D5691      BA0830E

In [61]:
# ---- Block 4: Add is_oversize column, clean OS text out of Bag No. (fixed) ----

for code, df in spectro_data.items():
    def parse_os(bag_val):
        if bag_val is None:
            return None, False
        bag_val = str(bag_val).strip()

        if bag_val.upper() == "OS":
            return None, True

        is_os = bag_val.upper().endswith(" OS")
        if is_os:
            bag_val = bag_val[:-3].strip()
        return bag_val, is_os

    parsed = df["Bag No."].apply(parse_os)
    df["Bag No."] = [p[0] for p in parsed]
    df["is_oversize"] = [p[1] for p in parsed]

    cols = list(df.columns)
    cols.remove("is_oversize")
    bag_idx = cols.index("Bag No.")
    cols.insert(bag_idx + 1, "is_oversize")
    spectro_data[code] = df[cols]

for code, df in spectro_data.items():
    print(f"\n--- {code} ---")
    print(df[["Color Simulation", "Product Code", "Name", "Bag No.", "is_oversize"]].to_string(index=False))


--- BA0830E ---
Color Simulation Product Code           Name Bag No.  is_oversize
       #FF2F5790      BA0830E STD 2260P 0.4%    None        False
       #FF2F568E      BA0830E LT 4484AL 0.4%    None        False
       #FF325489      BA0830E  DR 2193P 0.4%    None        False
       #FF2D5692      BA0830E         4484AL    None        False
       #FF2F5691      BA0830E         4528AL    None        False
       #FF325A94      BA0830E         4529AL    None        False
       #FF2E5792      BA0830E         4530AL    None        False
       #FF2F5690      BA0830E         4531AL    None        False
       #FF2E5591      BA0830E         4532AL    None        False
       #FF2C5693      BA0830E         4533AL    None        False
       #FF2E5692      BA0830E         4534AL    None        False
       #FF2E5793      BA0830E         4535AL    None        False
       #FF2D5692      BA0830E         4536AL    None        False
       #FF2D5591      BA0830E         4537AL    None       

### \- 3D Step 2 Sub of \(C\): Missing bag correction

In [62]:
# ---- 3D Step 2 (addendum): Manual correction — add missing Bag No. ----
# Ms. Jam confirmed: these 2 Spectro rows are missing their bag number due to
# operator input omission. Bag range added to match the corresponding QC record.

MANUAL_BAG_NO_CORRECTIONS = {
    ("RA16826E", "3700AO"): "1-2",
    ("WA15190E", "8910AN"): "248-249",
}

for code, df in spectro_data.items():
    for (target_code, target_lot), bag_value in MANUAL_BAG_NO_CORRECTIONS.items():
        if code != target_code:
            continue
        mask = (df["Name"] == target_lot) & (df["Bag No."].isna())
        if mask.any():
            df.loc[mask, "Bag No."] = bag_value
            print(f"{code}: set Bag No.='{bag_value}' for lot {target_lot} ({mask.sum()} row(s))")

RA16826E: set Bag No.='1-2' for lot 3700AO (1 row(s))
WA15190E: set Bag No.='248-249' for lot 8910AN (1 row(s))


### \- 3D Step 3: Check cleaned values

In [63]:
for code, df in spectro_data.items():
    print(f"\n--- {code} ({len(df)} rows) ---")
    print(df.to_string(index=False))


--- BA0830E (123 rows) ---
Color Simulation Product Code           Name Bag No.  is_oversize           Date Time ΔE*00    L*    C*     h°    a*     b*   ΔL*   ΔC*   ΔH*   Δa*   Δb*          Color Offset Judgement               Name_raw  is_reference_row
       #FF2F5790      BA0830E STD 2260P 0.4%    None        False 2024-11-20 11:42:51    -- 36.19 36.36 271.19  0.76 -36.35    --    --    --    --    --                    --        -- BA0830E 2260P 0.4% STD              True
       #FF2F568E      BA0830E LT 4484AL 0.4%    None        False 2024-11-20 11:42:54  0.24 35.93 35.92 271.05  0.66 -35.92 -0.26 -0.44 -0.09  -0.1  0.43                   NaN      Pass      4484AL 0.4% LIGHT              True
       #FF325489      BA0830E  DR 2193P 0.4%    None        False 2024-11-20 11:43:00  1.53 35.14 34.03 272.09  1.24 -34.00 -1.05 -2.33  0.55  0.48  2.34          Dark+, Blue-      Fail        2193P 0.4% DARK              True
       #FF2D5692      BA0830E         4484AL    None        Fals

### \- 3D Step 4: Sort spectro bag column

In [64]:
def bag_sort_key(bag_val):
    """Returns the starting number of a bag value for sorting (handles ranges and single OS)."""
    if bag_val is None or str(bag_val).strip() == "":
        return -1
    bag_val = str(bag_val).strip()
    if "-" in bag_val:
        try:
            return int(bag_val.split("-")[0])
        except:
            return 9999
    try:
        return int(bag_val)
    except:
        return 9999

for code, df in spectro_data.items():
    real_df = df[df["is_reference_row"] == False].copy()
    real_df["_bag_sort"] = real_df["Bag No."].apply(bag_sort_key)
    real_df_sorted = real_df.sort_values(by=["Name", "_bag_sort"]).drop(columns=["_bag_sort"])

    print(f"\n--- {code} ---")
    print(real_df_sorted[["Name", "Bag No.", "is_oversize"]].to_string(index=False))


--- BA0830E ---
  Name Bag No.  is_oversize
4484AL    None        False
4528AL    None        False
4529AL    None        False
4530AL    None        False
4531AL    None        False
4532AL    None        False
4533AL    None        False
4534AL    None        False
4535AL    None        False
4536AL    None        False
4537AL    None        False
4538AL    None        False
4539AL    None        False
4540AL    None        False
4541AL    None        False
5342AN    None        False
5343AN    None        False
5344AN    None        False
5345AN    None        False
5346AN    None        False
5347AN    None        False
5347AN       7         True
5801AN    None        False
5802AN    None        False
5803AN    None        False
5804AN    None        False
5805AN    None        False
5806AN    None        False
5807AN    None        False
5808AN    None        False
5809AN    None        False
5810AN    None        False
5811AN    None        False
5812AN    None        False
581

### \- 3D Step 5: Check spectro bag column gap

In [65]:
def parse_bag_range(bag_val):
    if bag_val is None or str(bag_val).strip() == "":
        return None
    bag_val = str(bag_val).strip()
    if "-" in bag_val:
        try:
            a, b = bag_val.split("-")
            return int(a), int(b)
        except:
            return None
    try:
        n = int(bag_val)
        return n, n
    except:
        return None

anomalies = []

for code, df in spectro_data.items():
    real_df = df[df["is_reference_row"] == False]

    for lot, group in real_df.groupby("Name"):
        ranges = []
        for bag_val in group["Bag No."]:
            r = parse_bag_range(bag_val)
            if r:
                ranges.append(r)
        ranges.sort()

        for i in range(1, len(ranges)):
            prev_end = ranges[i-1][1]
            curr_start = ranges[i][0]
            if curr_start <= prev_end:
                anomalies.append({"product_code": code, "lot": lot, "issue": "overlap",
                                   "prev_range": ranges[i-1], "curr_range": ranges[i]})
            elif curr_start > prev_end + 1:
                anomalies.append({"product_code": code, "lot": lot, "issue": "gap",
                                   "prev_range": ranges[i-1], "curr_range": ranges[i]})

if anomalies:
    print(f"FOUND: {len(anomalies)} bag range anomaly(ies)")
    for a in anomalies:
        print(f"  [{a['product_code']}] {a['lot']}: {a['issue']} between {a['prev_range']} and {a['curr_range']}")
else:
    print("No bag range anomalies found.")

FOUND: 1 bag range anomaly(ies)
  [RA18011E] 7815AN: overlap between (27, 29) and (29, 30)


TEST BLOCK: TRY APPLYING CORRECTIONS

In [66]:
# ---- Manual corrections for confirmed typos found in Block 2 ----
# Add entries here only after confirming an anomaly is a typo, not a real gap.
# Format: (product_code, lot_name, original_bag_value) -> corrected_bag_value

BAG_TYPO_CORRECTIONS = {
    ("RA18011E", "7815AN", "27-29"): "27-28",
}

for (code, lot, original_bag), corrected_bag in BAG_TYPO_CORRECTIONS.items():
    df = spectro_data[code]
    mask = (df["Name"] == lot) & (df["Bag No."] == original_bag)
    df.loc[mask, "Bag No."] = corrected_bag

print(f"Applied {len(BAG_TYPO_CORRECTIONS)} typo correction(s) to Spectro Bag No. values.")
for (code, lot, original_bag), corrected_bag in BAG_TYPO_CORRECTIONS.items():
    print(f"  [{code}] {lot}: '{original_bag}' -> '{corrected_bag}'")

Applied 1 typo correction(s) to Spectro Bag No. values.
  [RA18011E] 7815AN: '27-29' -> '27-28'


In [67]:
# ---- bag_no_corrected already built in Section A (2H), including remarks fallback.
# No recomputation here — this would overwrite the remarks-recovered values. ----
print(f"bag_no_corrected already set from Section A. Non-blank count: {qc_df['bag_no_corrected'].notna().sum()}")

bag_no_corrected already set from Section A. Non-blank count: 1206


### \- 3D Step 6: Validate spectro bag number gaps

In [68]:
# ---- Re-check for bag range anomalies after typo corrections, cross-verify against QC ----

def get_qc_bag_range(product_code, lot_name):
    """Returns (min_bag, max_bag) from QC's bag_no_corrected for this product_code + lot, or None."""
    matches = qc_df[
        (qc_df["product_code_corrected"] == product_code) &
        (qc_df["lot_number_corrected"] == lot_name) &
        (qc_df["bag_no_corrected"].notna())
    ]
    if matches.empty:
        return None

    all_bags = []
    for val in matches["bag_no_corrected"]:
        r = parse_bag_range(val)
        if r:
            all_bags.extend([r[0], r[1]])

    if not all_bags:
        return None
    return min(all_bags), max(all_bags)


confirmed_gaps = []

for code, df in spectro_data.items():
    real_df = df[df["is_reference_row"] == False]

    for lot, group in real_df.groupby("Name"):
        ranges = []
        for bag_val in group["Bag No."]:
            r = parse_bag_range(bag_val)
            if r:
                ranges.append(r)
        ranges.sort()

        for i in range(1, len(ranges)):
            prev_end = ranges[i-1][1]
            curr_start = ranges[i][0]

            if curr_start <= prev_end:
                print(f"WARNING: [{code}] {lot} still has an OVERLAP between {ranges[i-1]} and {ranges[i]} — needs manual review, not treated as gap.")
                continue

            if curr_start > prev_end + 1:
                missing_start, missing_end = prev_end + 1, curr_start - 1
                qc_range = get_qc_bag_range(code, lot)

                if qc_range and qc_range[0] <= missing_start and qc_range[1] >= missing_end:
                    confirmed_gaps.append({
                        "product_code": code, "lot": lot,
                        "missing_start": missing_start, "missing_end": missing_end,
                        "qc_range": qc_range
                    })
                else:
                    print(f"NOTE: [{code}] {lot} has a Spectro-side gap {missing_start}-{missing_end}, "
                          f"but QC does not cover this range (QC range found: {qc_range}) — not auto-filled, needs review.")

if confirmed_gaps:
    print(f"\nCONFIRMED GAPS (QC-verified, safe to fill): {len(confirmed_gaps)}")
    for g in confirmed_gaps:
        print(f"  [{g['product_code']}] {g['lot']}: missing bag(s) {g['missing_start']}-{g['missing_end']} (QC covers {g['qc_range']})")
else:
    print("\nNo confirmed gaps remaining.")


No confirmed gaps remaining.


In [69]:
# ---- Fill confirmed gaps with blank-Spectro-data rows ----

if not confirmed_gaps:
    print("No confirmed gaps to fill — skipping.")
else:
    spectro_only_cols = [c for c in df.columns]  # placeholder, refined per-file below

new_rows_by_code = {}

for gap in confirmed_gaps:
    code = gap["product_code"]
    lot = gap["lot"]
    df = spectro_data[code]

    template_row = df[(df["Name"] == lot) & (df["is_reference_row"] == False)].iloc[0].copy()

    new_row = template_row.copy()
    for col in df.columns:
        if col not in ["Color Simulation", "Product Code", "Name", "Bag No.", "is_oversize", "is_reference_row"]:
            new_row[col] = None
    new_row["Bag No."] = f'{gap["missing_start"]}-{gap["missing_end"]}' if gap["missing_start"] != gap["missing_end"] else str(gap["missing_start"])
    new_row["is_oversize"] = False
    new_row["match_status"] = "QC ONLY (NO SPECTRO READING)"
    new_rows_by_code.setdefault(code, []).append(new_row)

for code, rows in new_rows_by_code.items():
    spectro_data[code] = pd.concat([spectro_data[code], pd.DataFrame(rows)], ignore_index=True)

print(f"Gap-filled {sum(g['missing_end'] - g['missing_start'] + 1 for g in confirmed_gaps)} row(s) across {len(confirmed_gaps)} confirmed gap(s).")

No confirmed gaps to fill — skipping.
Gap-filled 0 row(s) across 0 confirmed gap(s).


# \(C\) MATCH & MERGE

## 4\. Build flat QC lookup 

In [70]:
# ---- Step 4: Build one flat QC lookup table ----

# TERMINOLOGY:
#   Mode A — Default matching. Spectro's 'Name' column holds the regular sticker
#            lot number, matched directly against QC's lot_number.
#   Mode B — Special matching. Spectro's 'Name' column holds the INTERNAL lot
#            number instead of the sticker lot number, matched against QC's
#            internal_lot (or the parenthesis-fallback if internal_lot is blank).
#            Applies only to specific clients: PACKAGEWORLD/ABI, ROWELL,
#            DYNAMICCAPS/NICE — see MODE_B_CODES below.

MODE_B_CODES = {"BA12556E", "WA12282E", "WA15151E", "WA15816E",
                "BA17042E", "BA17070E",
                "WA7997E", "WA15218E", "WA15229E"}

QC_COLUMNS_TO_MERGE = ["evaluated_on", "evaluated_by", "customer", "status", "formula_id"]

# OS_REMARKS_PATTERN = re.compile(r"(from\s*os|os)", re.IGNORECASE)
OS_REMARKS_PATTERN = re.compile(r"\b(from\s+os|os)\b", re.IGNORECASE)

def check_os_in_remarks(remarks_value):
    if pd.isna(remarks_value):
        return False
    return bool(OS_REMARKS_PATTERN.search(str(remarks_value)))

def get_internal_lot_key(row):
    """Mode B key: internal_lot column first, then parenthesis-inner fallback."""
    internal_lot = row.get("internal_lot")
    if not pd.isna(internal_lot) and str(internal_lot).strip() != "":
        return str(internal_lot).strip().upper()

    lot_number = str(row["lot_number_corrected"]).strip()
    match = re.match(r"^.*?\(([A-Za-z0-9]+)\)$", lot_number)
    if match:
        inner = match.group(1).strip()
        if re.search(r"[A-Za-z]", inner):
            return inner.upper()
    return None

# ---- Helper: normalize any bag value to canonical 'min-max' or single 'N' form ----
def normalize_bag_range(bag_val):
    if bag_val is None or pd.isna(bag_val) or str(bag_val).strip() == "":
        return None
    bag_val = str(bag_val).strip()
    if "-" in bag_val:
        try:
            a, b = bag_val.split("-")
            a, b = int(a), int(b)
            lo, hi = min(a, b), max(a, b)
            return str(lo) if lo == hi else f"{lo}-{hi}"
        except:
            return bag_val
    try:
        return str(int(bag_val))
    except:
        return bag_val

qc_lookup_rows = []

for idx, row in qc_df.iterrows():
    product_code = row["product_code_corrected"]
    is_mode_b = product_code in MODE_B_CODES
    lot_format = row["lot_format"]
    lot_value = str(row["lot_number_corrected"]).strip()

    base_entry = {
        "id": int(row["id"]),
        "product_code": product_code,
        "mode": "B" if is_mode_b else "A",
        "remarks": row.get("remarks"),
    }
    for col in QC_COLUMNS_TO_MERGE:
        base_entry[col] = row.get(col)

    if is_mode_b:
        key = get_internal_lot_key(row)
        lot_key_value = key if key else lot_value.upper()
        bag_key_from_qc = (
            normalize_bag_range(row.get("bag_no_corrected"))
            if pd.notna(row.get("bag_no_corrected")) else None
        )

        # Entry 1: bag-less — matches a real physical Spectro reading (no bag suffix),
        # unchanged from before.
        entry_no_bag = dict(base_entry)
        entry_no_bag["lot_key"] = lot_key_value
        entry_no_bag["bag_key"] = None
        qc_lookup_rows.append(entry_no_bag)

        # Entry 2: bag-keyed — matches a "missing lot" placeholder row (Block 6),
        # which carries QC's own bag_no as its Bag Number since Spectro never measured it.
        if bag_key_from_qc is not None:
            entry_with_bag = dict(base_entry)
            entry_with_bag["lot_key"] = lot_key_value
            entry_with_bag["bag_key"] = bag_key_from_qc
            qc_lookup_rows.append(entry_with_bag)

        continue

    # Mode A
    if lot_format == "single":
        entry = dict(base_entry)
        entry["lot_key"] = lot_value.upper()
        entry["bag_key"] = None
        qc_lookup_rows.append(entry)

    elif lot_format == "range":
        parts = lot_value.split("-")
        if len(parts) == 2:
            expanded = expand_range(parts[0].strip(), parts[1].strip())
            if expanded:
                for lot in expanded:
                    entry = dict(base_entry)
                    entry["lot_key"] = lot.upper()
                    entry["bag_key"] = None
                    qc_lookup_rows.append(entry)

    elif lot_format == "parenthesis":
        # bag_single_paren: 1234AB(5)
        m_single = re.match(r"^(\d{4}[A-Za-z]{1,2})\((\d+)\)$", lot_value)
        # bag_range_paren: 1234AB(5-10) -> expand into individual bags
        m_range = re.match(r"^(\d{4}[A-Za-z]{1,2})\((\d+)-(\d+)\)$", lot_value)
        # internal_lot_paren handled only under Mode B above; skip here for Mode A

        if m_single:
            entry = dict(base_entry)
            entry["lot_key"] = m_single.group(1).upper()
            entry["bag_key"] = m_single.group(2)
            qc_lookup_rows.append(entry)

        elif m_range:
            base_lot = m_range.group(1).upper()
            bag_start, bag_end = int(m_range.group(2)), int(m_range.group(3))
            entry = dict(base_entry)
            entry["lot_key"] = base_lot
            entry["bag_key"] = normalize_bag_range(f"{bag_start}-{bag_end}")
            qc_lookup_rows.append(entry)

qc_lookup_df = pd.DataFrame(qc_lookup_rows)

print(f"Total QC lookup rows built: {len(qc_lookup_df)}")
print(qc_lookup_df.groupby("mode").size())
# print(qc_lookup_df.head(100).to_string(index=False))

Total QC lookup rows built: 30376
mode
A    29469
B      907
dtype: int64


In [71]:
# ---- Block 6: Add missing Mode B internal lots that have no Spectro reading at all ----
# Mode B lots are sometimes never measured in Spectro at all, since Spectro sampling
# alternates through lots rather than reading every one sequentially. This catches lots
# (including the anchor lot itself, e.g. "8348AM") that exist in QC but have zero rows
# in Spectro, and adds a placeholder row so they still show up in the final report.
#
# NOTE: Intentionally scoped to Mode B only for now — this is the only case raised so far.
# If Ms. Jam wants this same check for Mode A lots later, reuse this block and drop the
# `if code not in MODE_B_CODES: continue` guard below.

missing_lot_rows_by_code = {}

for code, df in spectro_data.items():
    if code not in MODE_B_CODES:
        continue  # Mode A not covered yet — see note above

    existing_lots_in_spectro = set(
        df.loc[~df["is_reference_row"], "Name"].astype(str).str.strip().str.upper()
    )

    qc_subset = qc_df[qc_df["product_code_corrected"] == code]
    seen_missing_lots = set()

# ---- Build family ranges from Spectro's own lots (per suffix, clustered by gap <= 2) ----
    def parse_lot(lot):
        m = re.match(r"^(\d+)([A-Za-z]+)$", lot)
        if not m:
            return None
        return int(m.group(1)), m.group(2).upper()

    parsed_spectro = [parse_lot(lot) for lot in existing_lots_in_spectro]
    parsed_spectro = [p for p in parsed_spectro if p]

    by_suffix = {}
    for num, suffix in parsed_spectro:
        by_suffix.setdefault(suffix, []).append(num)

    families = []  # (suffix, min_num, max_num)
    for suffix, nums in by_suffix.items():
        nums = sorted(set(nums))
        cluster = [nums[0]]
        for n in nums[1:]:
            if n - cluster[-1] <= 2:
                cluster.append(n)
            else:
                families.append((suffix, cluster[0], cluster[-1]))
                cluster = [n]
        families.append((suffix, cluster[0], cluster[-1]))

    def within_a_family(num, suffix):
        return any(s == suffix and lo <= num <= hi for s, lo, hi in families)

    for _, qc_row in qc_subset.iterrows():
        key = get_internal_lot_key(qc_row)
        if key:
            lot_value = key  # child lot — from internal_lot column OR parenthesis extraction
        else:
            lot_value = str(qc_row["lot_number_corrected"]).strip().upper()  # anchor lot

        if lot_value in existing_lots_in_spectro or lot_value in seen_missing_lots:
            continue

        parsed = parse_lot(lot_value)
        if not parsed or not within_a_family(*parsed):
            continue  # outside every Spectro family's range — not needed this batch

        seen_missing_lots.add(lot_value)

        new_row = df.iloc[0].copy()
        for col in df.columns:
            new_row[col] = None
        new_row["Product Code"] = code
        new_row["Name"] = lot_value
        new_row["Bag No."] = qc_row.get("bag_no_corrected")   # from QC's own bag_no column
        # new_row["is_oversize"] = False
        new_row["is_oversize"] = check_os_in_remarks(qc_row.get("remarks"))
        new_row["is_reference_row"] = False
        new_row["match_status"] = "QC ONLY (NO SPECTRO READING)"

        missing_lot_rows_by_code.setdefault(code, []).append(new_row)

for code, rows in missing_lot_rows_by_code.items():
    spectro_data[code] = pd.concat([spectro_data[code], pd.DataFrame(rows)], ignore_index=True)

total_added = sum(len(rows) for rows in missing_lot_rows_by_code.values())
print(f"Added {total_added} missing-lot placeholder row(s) across {len(missing_lot_rows_by_code)} Mode B file(s).")
for code, rows in missing_lot_rows_by_code.items():
    for r in rows:
        print(f"  [{code}] {r['Name']} — Bag No. {r['Bag No.']}")

Added 21 missing-lot placeholder row(s) across 1 Mode B file(s).
  [WA12282E] 5145AN — Bag No. 170
  [WA12282E] 5143AN — Bag No. 130
  [WA12282E] 5141AN — Bag No. 90
  [WA12282E] 5139AN — Bag No. 50
  [WA12282E] 3055AN — Bag No. 110
  [WA12282E] 3053AN — Bag No. 70
  [WA12282E] 3051AN — Bag No. 30
  [WA12282E] 2767AN — Bag No. 30
  [WA12282E] 8357AM — Bag No. 190
  [WA12282E] 8355AM — Bag No. 150
  [WA12282E] 8353AM — Bag No. 110
  [WA12282E] 8351AM — Bag No. 70
  [WA12282E] 8349AM — Bag No. 30
  [WA12282E] 6247AM — Bag No. 150
  [WA12282E] 6245AM — Bag No. 110
  [WA12282E] 6243AM — Bag No. 70
  [WA12282E] 6241AM — Bag No. 30
  [WA12282E] 4407AM — Bag No. 150
  [WA12282E] 4405AM — Bag No. 110
  [WA12282E] 4403AM — Bag No. 70
  [WA12282E] 4401AM — Bag No. 30


C:\Users\Administrator\AppData\Local\Temp\ipykernel_20004\2169205430.py:83: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  spectro_data[code] = pd.concat([spectro_data[code], pd.DataFrame(rows)], ignore_index=True)


## 4A\. Restrict bag expansion to Mode A only

In [72]:
# ---- Step 4 (addendum) FIX: only expand bag_no_corrected for Mode A entries ----

additional_lookup_rows = []
rows_to_remove_indices = []

for i, entry in enumerate(qc_lookup_rows):
    if entry["mode"] != "A":   # <-- the fix: skip Mode B entirely
        continue

    qc_row_id = entry["id"]
    matching_qc_row = qc_df[qc_df["id"] == qc_row_id]
    if matching_qc_row.empty:
        continue
    qc_row = matching_qc_row.iloc[0]

    lot_format = qc_row["lot_format"]
    bag_no_val = qc_row.get("bag_no_corrected")

    if entry["bag_key"] is None and lot_format in ("single", "range") \
            and bag_no_val is not None and str(bag_no_val).strip() != "":

        new_entry = dict(entry)
        new_entry["bag_key"] = normalize_bag_range(bag_no_val)
        additional_lookup_rows.append(new_entry)

        rows_to_remove_indices.append(i)

qc_lookup_rows = [e for i, e in enumerate(qc_lookup_rows) if i not in rows_to_remove_indices]
qc_lookup_rows.extend(additional_lookup_rows)

qc_lookup_df = pd.DataFrame(qc_lookup_rows)

print(f"Additional bag-expanded entries added: {len(additional_lookup_rows)}")
print(f"Total QC lookup rows now: {len(qc_lookup_df)}")
print(qc_lookup_df.groupby("mode").size())

Additional bag-expanded entries added: 761
Total QC lookup rows now: 30376
mode
A    29469
B      907
dtype: int64


## 4B\. Clean NONE/NAN artifacts from lookup keys

In [73]:
# ---- Fix: Clean qc_lookup_df's bag_key and lot_key to remove text "NONE"/"NAN" artifacts ----
qc_lookup_df["bag_key"] = qc_lookup_df["bag_key"].replace({"NONE": None, "NAN": None, "nan": None})
qc_lookup_df["lot_key"] = qc_lookup_df["lot_key"].replace({"NONE": None, "NAN": None, "nan": None})

print(qc_lookup_df["bag_key"].isna().sum(), "bag_key entries are now real None")

28338 bag_key entries are now real None


## 4C\. Match Spectro rows against QC lookup

In [74]:
# ---- Step 4A: Matching loop ----
# Matches each real Spectro row against qc_lookup_df using (product_code, lot_key, bag_key, mode).
# Reference rows are skipped from matching entirely.
# Oversize cross-check: if Spectro row is flagged is_oversize=True, check the matched
# QC row's remarks for "OS" or "from OS" (any case). This does not affect the match itself.

#OS_REMARKS_PATTERN = re.compile(r"(from\s*os|os)", re.IGNORECASE)

#def check_os_in_remarks(remarks_value):
#    if pd.isna(remarks_value):
#        return False
#    return bool(OS_REMARKS_PATTERN.search(str(remarks_value)))


all_matched_rows = []

for code, df in spectro_data.items():
    is_mode_b = code in MODE_B_CODES
    mode_label = "B" if is_mode_b else "A"

    lookup_subset = qc_lookup_df[
        (qc_lookup_df["product_code"] == code) & (qc_lookup_df["mode"] == mode_label)
    ]

    matched = unmatched = skipped_ref = 0

    for idx, row in df.iterrows():
        is_gap_filled = row.get("match_status") == "QC ONLY (NO SPECTRO READING)"

        result_row = row.copy()
        for col in QC_COLUMNS_TO_MERGE:
            result_row[col] = None
        result_row["match_status"] = None
        result_row["Oversize"] = bool(row.get("is_oversize", False))
        result_row["os_remarks_match"] = None

        if row["is_reference_row"]:
            result_row["match_status"] = "REFERENCE ROW"
            skipped_ref += 1
            all_matched_rows.append(result_row)
            continue

        lot_key = str(row["Name"]).strip().upper()
        spectro_bag = normalize_bag_range(row["Bag No."])
        spectro_range = parse_bag_range(spectro_bag) if spectro_bag else None  # (lo, hi) or None

        candidates = lookup_subset[lookup_subset["lot_key"] == lot_key]

        if spectro_range is not None:
            def qc_covers_spectro(qc_bag_val):
                qc_range = parse_bag_range(qc_bag_val)
                if qc_range is None:
                    return False
                return qc_range[0] <= spectro_range[0] and qc_range[1] >= spectro_range[1]

            candidates = candidates[candidates["bag_key"].apply(qc_covers_spectro)]
        else:
            candidates = candidates[candidates["bag_key"].isna()]

        if candidates.empty:
            result_row["match_status"] = "NO MATCH FOUND"
            unmatched += 1
            all_matched_rows.append(result_row)
            continue

        if len(candidates) > 1:
            candidates = candidates.copy()
            candidates["evaluated_on_dt"] = pd.to_datetime(candidates["evaluated_on"], errors="coerce")
            candidates = candidates.sort_values("evaluated_on_dt", ascending=False)
            best_match = candidates.iloc[0]
            result_row["match_status"] = f"MATCHED (latest of {len(candidates)})"
        else:
            best_match = candidates.iloc[0]
            result_row["match_status"] = "MATCHED"

        if is_gap_filled:
            result_row["match_status"] = "QC ONLY (NO SPECTRO READING)"

        for col in QC_COLUMNS_TO_MERGE:
            result_row[col] = best_match[col]

        if result_row["Oversize"]:
            result_row["os_remarks_match"] = check_os_in_remarks(best_match["remarks"])

        matched += 1
        all_matched_rows.append(result_row)

    print(f"{code} (Mode {mode_label}): Matched={matched}, Unmatched={unmatched}, Reference={skipped_ref}")

final_df = pd.DataFrame(all_matched_rows)
print(f"\nTotal rows: {len(final_df)}")
print(final_df["match_status"].value_counts())

BA0830E (Mode A): Matched=120, Unmatched=0, Reference=3
BA11939E (Mode A): Matched=2, Unmatched=0, Reference=3
BA11959E (Mode A): Matched=2, Unmatched=0, Reference=3
BA12013E (Mode A): Matched=15, Unmatched=0, Reference=3
BA12068E (Mode A): Matched=21, Unmatched=0, Reference=3
BA12615E (Mode A): Matched=56, Unmatched=0, Reference=6
BA12618E (Mode A): Matched=36, Unmatched=0, Reference=3
BA12855E (Mode A): Matched=1, Unmatched=0, Reference=3
BA12861E (Mode A): Matched=35, Unmatched=0, Reference=3
BA13023E (Mode A): Matched=100, Unmatched=0, Reference=6
BA13652E (Mode A): Matched=3, Unmatched=0, Reference=3
BA14168E (Mode A): Matched=3, Unmatched=0, Reference=3
BA14304E (Mode A): Matched=62, Unmatched=0, Reference=3
BA14432E (Mode A): Matched=328, Unmatched=0, Reference=3
BA14456E (Mode A): Matched=97, Unmatched=0, Reference=3
BA15197E (Mode A): Matched=144, Unmatched=0, Reference=4
BA15494E (Mode A): Matched=51, Unmatched=0, Reference=6
BA15965E (Mode A): Matched=139, Unmatched=0, Refer

In [75]:
# ---- Step 4A (diagnostic): List all NO MATCH FOUND rows ----

no_match_rows = final_df[final_df["match_status"] == "NO MATCH FOUND"]

print(f"Total NO MATCH FOUND rows: {len(no_match_rows)}")
print(no_match_rows[["Product Code", "Name", "Bag No.", "is_oversize"]].to_string(index=False))

Total NO MATCH FOUND rows: 2
Product Code   Name Bag No.  is_oversize
    BA17855E 0670AO       1         True
    BA17936E 1540AN    None        False


# \(D\) OUTPUT

## 5\. Rename columns

In [76]:
final_df = final_df.rename(columns={
    "Name": "Lot Number",
    "Bag No.": "Bag Number",
    "Judgement": "Spectro Judgement",
    "status": "Final QC Status",
    "evaluated_on": "Evaluated On",
    "evaluated_by": "Evaluated By",
    "customer": "Customer",
    "formula_id": "Formula ID",
    "match_status": "Match Status",
})

## 5A\. Clean value formats

In [77]:
def clean_upper_nospace(value):
    """Uppercase, strip all spaces. Leaves real blanks untouched."""
    if pd.isna(value) or str(value).strip() == "":
        return value
    return str(value).strip().upper().replace(" ", "")

final_df["Evaluated By"] = final_df["Evaluated By"].apply(clean_upper_nospace)
final_df["Customer"] = final_df["Customer"].apply(clean_upper_nospace)

# Oversize -> real boolean
final_df["Oversize"] = final_df["Oversize"].astype(bool)

# Formula ID -> integer, not float (fixes 17026.0 issue). Int64 (capital I)
# keeps it a nullable-int type so blanks stay blank instead of forcing 0.
final_df["Formula ID"] = pd.to_numeric(final_df["Formula ID"], errors="coerce").astype("Int64")

## 5B\. Column order

In [78]:
# ---- Define final output column order ----

FINAL_COLUMN_ORDER = [
    "Color Simulation", "Lot Number", "Product Code", "Bag Number", "Oversize",
    "Date Time", "ΔE*00", "L*", "C*", "h°", "a*", "b*",
    "ΔL*", "ΔC*", "ΔH*", "Δa*", "Δb*", "Color Offset",
    "Spectro Judgement", "Final QC Status", "Evaluated On", "Evaluated By",
    "Customer", "Formula ID", "Match Status",
]

final_df = final_df[FINAL_COLUMN_ORDER]
print(final_df.columns.tolist())
print(final_df.head(10))

['Color Simulation', 'Lot Number', 'Product Code', 'Bag Number', 'Oversize', 'Date Time', 'ΔE*00', 'L*', 'C*', 'h°', 'a*', 'b*', 'ΔL*', 'ΔC*', 'ΔH*', 'Δa*', 'Δb*', 'Color Offset', 'Spectro Judgement', 'Final QC Status', 'Evaluated On', 'Evaluated By', 'Customer', 'Formula ID', 'Match Status']
  Color Simulation      Lot Number Product Code Bag Number  Oversize  \
0        #FF2F5790  STD 2260P 0.4%      BA0830E       None     False   
1        #FF2F568E  LT 4484AL 0.4%      BA0830E       None     False   
2        #FF325489   DR 2193P 0.4%      BA0830E       None     False   
3        #FF2D5692          4484AL      BA0830E       None     False   
4        #FF2F5691          4528AL      BA0830E       None     False   
5        #FF325A94          4529AL      BA0830E       None     False   
6        #FF2E5792          4530AL      BA0830E       None     False   
7        #FF2F5690          4531AL      BA0830E       None     False   
8        #FF2E5591          4532AL      BA0830E       None

## 5C: Sort: reference rows first per product code group

In [79]:
# ---- Step 5C: Sort — reference rows first, then by lot number, then bag ascending ----

final_df["_ref_sort"] = (final_df["Match Status"] != "REFERENCE ROW").astype(int)
# 0 = reference row (sorts first), 1 = everything else

def bag_sort_key(bag_val):
    """Numeric sort key for Bag Number. Ranges like '1-2' sort by their start number."""
    if pd.isna(bag_val) or str(bag_val).strip() == "":
        return -1  # blanks (e.g. reference rows) sort first
    bag_str = str(bag_val).strip()
    if "-" in bag_str:
        try:
            return int(bag_str.split("-")[0])
        except:
            return 0
    try:
        return int(bag_str)
    except:
        return 0

final_df["_bag_sort"] = final_df["Bag Number"].apply(bag_sort_key)

final_df = final_df.sort_values(
    by=["Product Code", "_ref_sort", "Lot Number", "_bag_sort"],
    kind="stable"
).drop(columns=["_ref_sort", "_bag_sort"]).reset_index(drop=True)

## 5D\. Gray formatting for reference rows

In [80]:
HEADER_FILL = PatternFill(start_color="2F4F4F", end_color="2F4F4F", fill_type="solid")
HEADER_FONT = Font(color="FFFFFF", size=12, bold=True)

FAIL_FILL = PatternFill(start_color="F9E8BD", end_color="F9E8BD", fill_type="solid")
REF_FILL = PatternFill(start_color="D9D9D9", end_color="D9D9D9", fill_type="solid")
HEADER_ALIGNMENT = Alignment(horizontal="center", vertical="center", wrap_text=False)
QC_ONLY_FILL = PatternFill(start_color="C6EFCE", end_color="C6EFCE", fill_type="solid")

def apply_output_formatting(ws, df):
    # 1. Header row styling
    for col_idx in range(1, len(df.columns) + 1):
        cell = ws.cell(row=1, column=col_idx)
        cell.fill = HEADER_FILL
        cell.font = HEADER_FONT
        cell.alignment = HEADER_ALIGNMENT

    # 2 & 3. Per-row styling (data rows start at Excel row 2)
    for row_idx, row in df.iterrows():
        excel_row = row_idx + 2
        is_ref = row["Match Status"] == "REFERENCE ROW"
        is_fail = str(row["Final QC Status"]).strip().upper() == "FAILED"
        is_qc_only = row["Match Status"] == "QC ONLY (NO SPECTRO READING)"

        for col_idx in range(1, len(df.columns) + 1):
            cell = ws.cell(row=excel_row, column=col_idx)

            if is_fail:
                cell.fill = FAIL_FILL
                cell.font = Font(bold=True)
            elif is_ref:
                cell.fill = REF_FILL
                cell.font = Font(bold=True)
            elif is_qc_only:
                cell.fill = QC_ONLY_FILL

## 5E\. Write final Excel file

In [81]:
ph_now = datetime.now(ZoneInfo("Asia/Manila"))

ph_now = datetime.now(ZoneInfo("Asia/Manila"))

VERSION_COUNTER_FILE = "output/.version_counter"
os.makedirs("output", exist_ok=True)
if os.path.exists(VERSION_COUNTER_FILE):
    with open(VERSION_COUNTER_FILE, "r") as f:
        VERSION_LABEL = int(f.read().strip()) + 1
else:
    VERSION_LABEL = 1
with open(VERSION_COUNTER_FILE, "w") as f:
    f.write(str(VERSION_LABEL))

OUTPUT_PATH = f"output/{VERSION_LABEL} - Final Merge - {ph_now.strftime('%B %d, %Y - %H_%M')}.xlsx"

ref_mask = final_df["Match Status"] == "REFERENCE ROW"
qc_only_mask = final_df["Match Status"] == "QC ONLY (NO SPECTRO READING)"

sheets = {
    "RAW_DATA": final_df,
    "COMPLETE_DATA": final_df[~ref_mask & ~qc_only_mask].reset_index(drop=True),
    "REFERENCE_DATA": final_df[ref_mask].reset_index(drop=True),
    "INCOMPLETE_DATA": final_df[qc_only_mask].reset_index(drop=True),
}

with pd.ExcelWriter(OUTPUT_PATH, engine="openpyxl") as writer:
    for sheet_name, df in sheets.items():
        df.to_excel(writer, index=False, sheet_name=sheet_name)
        ws = writer.sheets[sheet_name]
        apply_output_formatting(ws, df)

        for col_idx, col_name in enumerate(df.columns, start=1):
            max_len = max(
                df[col_name].astype(str).map(len).max() if len(df) else 0,
                len(str(col_name))
            )
            ws.column_dimensions[ws.cell(row=1, column=col_idx).column_letter].width = max_len + 4

        ws.row_dimensions[1].height = 22

print(f"Saved: {OUTPUT_PATH}")
print(f"Total rows: {len(final_df)}")
for sheet_name, df in sheets.items():
    print(f"  {sheet_name}: {len(df)} rows")

Saved: output/11 - Final Merge - July 24, 2026 - 10_47.xlsx
Total rows: 2299
  RAW_DATA: 2299 rows
  COMPLETE_DATA: 2168 rows
  REFERENCE_DATA: 110 rows
  INCOMPLETE_DATA: 21 rows


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=3b53bd28-7e59-47fa-80c2-b8dc99257c16' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>